In [7705]:
#| default_exp latex.divide

In [7706]:
#| export
from itertools import product
from os import PathLike
import re
from typing import Optional, TypedDict, Union

from pylatexenc.latexwalker import (
    LatexWalker, LatexEnvironmentNode, LatexMacroNode, LatexNode
)

from pylatexenc.latex2text import (
    LatexNodes2Text
)
import regex

from trouver.helper.files_and_folders import text_from_file

from trouver.helper.latex.comments import remove_comments
from trouver.latex.formatting import replace_commands_in_latex_document, replace_input_and_include, custom_commands
from trouver.latex.preamble import divide_preamble, replace_inclusion_of_style_file_with_code

In [7707]:
from fastcore.test import ExceptionExpected, test_eq

from trouver.helper.tests import _test_directory# , non_utf8_chars_in_file

In [7708]:
#| export
# matches `\newtheorem{theorem}{Theorem}`, `\newtheorem{proposition}[theorem]{Proposition}`
# does not match `\newtheorem{theorem}{Theorem}[Section]`

# SECOND_PARAMETER_PATTERN = re.compile(
#     # r'\\newtheorem\s*\{\s*(\w+)\s*\}\s*(\[\s*(\w+)\s*\])?\s*\{\s*(.*)\s*\}')
#     r'\\newtheorem\s*\{\s*(\w+)\s*\}\s*(\[\s*(\w+)\s*\])?\s*\{\s*(.*)\s*\}(?!\s*\[\s*(\w+)\s*\])')
SECOND_PARAMETER_PATTERN = regex.compile(
    r'\\newtheorem\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}'
    r'\s*(\[\s*(\w+)\s*\])?\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}'
    r'(?!\s*\[\s*(\w+)\s*\])',
    regex.MULTILINE)

SECOND_PARAMETER_PATTERN_WITH_OPTIONAL_STAR = regex.compile(
    r'\\newtheorem\*?\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}'
    r'\s*(\[\s*(\w+)\s*\])?\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}'
    r'(?!\s*\[\s*(\w+)\s*\])',
    regex.MULTILINE)

# matches `\newtheorem{theorem}{Theorem}`, `\newtheorem{theorem}{Theorem}[Section]`,
# does not match `\newtheorem{proposition}[theorem]{Proposition}`
# THIRD_PARAMETER_PATTERN = re.compile(
#     r'\\newtheorem\s*\{\s*(\w+)\s*\}\s*\{\s*(.*)\s*\}\s*(\[\s*(\w+)\s*\])?')
THIRD_PARAMETER_PATTERN = regex.compile(
    r'\\newtheorem\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}'
    r'\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}'
    r'\s*'
    r'(\[\s*(\w+)\s*\])?',
    regex.MULTILINE)

THIRD_PARAMETER_PATTERN_WITH_OPTIONAL_STAR = regex.compile(
    r'\\newtheorem\*?\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}'
    r'\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}'
    r'\s*'
    r'(\[\s*(\w+)\s*\])?',
    regex.MULTILINE)

# matches \section{title}, \subsection{title}, \subsubsection{title}, \section*{title}, etc.
# SECTION_LIKE_PATTERN = regex.compile(
#         r'\\(?:section|subsection|subsubsection)\s*(?:\[.*\])?(\*)?\s*'
#         r'\{((?>[^{}]+|\{(?2)\})*)\}',
#         regex.MULTILINE)
# Updated to include part
SECTION_LIKE_PATTERN = regex.compile(
    r'\\(?:part|section|subsection|subsubsection)\s*(?:\[.*\])?(\*)?\s*'
    r'\{((?>[^{}]+|\{(?2)\})*)\}',
    regex.MULTILINE)
    

# matches \begin{theorem},
ENVIRONMENT_PATTERN = regex.compile(
        r'\\begin\s*'
        r'\{((?>[^{}]+|\{(?1)\})*)\}',
        regex.MULTILINE)

## Divide LaTeX file into parts

To make Obsidian notes from a LaTeX file, I use sections/subsections, and environments as places to make new notes.

Things to think about:
Sections/subsections
environments, including theorems, corollaries, propositions, lemmas, definitions, notations
citations
Macros defined in the preamble?

LatexMacroNodes include: sections/subsections, citations, references, and labels, e.g.

```latex
> \section{Introduction}
\cite{ellenberg2nilpotent}
\subsection{The section conjecture}
\'e
\ref{fundamental-exact-sequence}
\cite{stix2010period}
\ref{fundamental-exact-sequence}
\cite{stix2012rational}
\cite[Appendix C]{stix2010period}
\subsection{The tropical section conjecture}
\label{subsec:tropical-section-conjecture}
```

#### Get the Document Node

In [7709]:
#| export
class NoDocumentNodeError(Exception):
    """Exception raised when a LatexEnvironmentNode corresponding to the document 
    environment is expected in a LaTeX string, but no such node exists.
    
    **Attributes**
    - text - str
        - The text in which the document environment is not found.
    """
    
    def __init__(self, text):
        self.text = text
        super().__init__(
            f"The following text does not contain a document environment:\n{text}")


In [7710]:
#| export
def find_document_node(
        text: str, # LaTeX str
        document_environment_name: str = "document" # The name of the document environment.
        ) -> LatexEnvironmentNode:
    """Find the `LatexNode` object for the main document in `text`.
    
    **Raises**
    - NoDocumentNodeError
        - If document environment node is not detected.
    """
    w = LatexWalker(text)
    nodelist, _, _ = w.get_latex_nodes(pos=0)
    for node in nodelist:
        if node.isNodeType(LatexEnvironmentNode)\
                and node.environmentname == document_environment_name:
            return node
    raise NoDocumentNodeError(text)

The main content of virtually all LaTeX math articles belongs to a document environment, which pylatexenc can often detect. The `find_document_node` function returns this `LatexEnvironmentNode` object:

In [7711]:
latex_file_path = _test_directory() / 'latex_examples' / 'latex_example_1' / 'main.tex'
text = text_from_file(latex_file_path)
document_node = find_document_node(text)

If the LaTeX file has no `document` environment, then a `NoDocumentNodeError` is raised:

In [7712]:
# This latex document has its `document` environment commented out.
latex_file_path = _test_directory() / 'latex_examples' / 'latex_example_2' / 'main.tex'
text = text_from_file(latex_file_path)
with ExceptionExpected(NoDocumentNodeError):
    document_node = find_document_node(text)

At the time of this writinga `NoDocumentNodeError` may be raised even if the LaTeX file has a proper `document` environment

In [7713]:
latex_file_path = _test_directory() / 'latex_examples' / 'example_with_a_command_with_begin.tex'
text = text_from_file(latex_file_path)

# Perhaps in the future, pylatexenc will be able to find the document node for this file.
# When that time comes, delete this example.
with ExceptionExpected(NoDocumentNodeError):
    find_document_node(text)


The `divide_preamble` function can be used to circumvent this problem:

In [7714]:
preamble, document = divide_preamble(text)
document_node = find_document_node(document)
test_eq(document_node.environmentname, 'document')
assert document_node.isNodeType(LatexEnvironmentNode)

In [7715]:
#| include: false
# Find no document node error causes

# latex_file_path = r'_tests\latex_full\litt_cfag\main.tex'
# text = text_from_file(latex_file_path)
# document_node = find_document_node(text)

### Detect environment names used in a file

In [7716]:
#| export
def environment_names_used(
        text: str # LaTeX document
        ) -> set[str]: # The set of all environment names used in the main document.
    """Return the set of all environment names used in the main document
    of the latex code.
    """
    document_node = find_document_node(text)
    return {node.environmentname for node in document_node.nodelist
            if node.isNodeType(LatexEnvironmentNode)}        

Writers often use different environment names. For examples, writers often use `theorem`, `thm`, or `theo` for theorem environments or `lemma` or `lem` for lemma environments. The `environment_names_used` function returns the environment names actually used in the tex file.

In the example below, note that only the environments that are actually used are returned. For instance, the preamble of the document defines the theorem environments `problem`, and `lemma` (among other things), but these are not actually used in the document itself.

In [7717]:
latex_file_path = _test_directory() / 'latex_examples' / 'has_fully_written_out_environment_names.tex'
sample_text_1 = text_from_file(latex_file_path)
sample_output_1 = environment_names_used(sample_text_1)
test_eq({'corollary', 'proof', 'maincorollary', 'abstract', 'proposition'}, sample_output_1)

The document in the example below uses shorter names for theorem environments:

In [7718]:
latex_file_path = _test_directory() / 'latex_examples' / 'has_shorter_environment_names.tex'
sample_text_2 = text_from_file(latex_file_path)
sample_output_2 = environment_names_used(sample_text_2)
test_eq({'conj', 'notation', 'corollary', 'defn'}, sample_output_2)

#### Identify the numbering convention of a LaTeX document

LaTeX documents have various number conventions. Here are some examples of papers on the arXiv and notes on their numbering schemes. Note that the source code to these articles are publicly available on the arXiv. 

- Ellenberg, Venkatesh, and Westerland, *[Homological stability for Hurwitz spaces and the Cohen-Lenstra conjecture over function fields](https://arxiv.org/abs/0912.0325)*, 
    - The subsections and theorem-like environments of each section share a numbering scheme, e.g. section 1 has subsection `1.1 The Cohen-Lenstra heuristics`, `1.2 Theorem`, `1.3 Hurwitz spaces`. This is accomplished by defining theorem-like environments using the `subsection` counter, e.g.

        ```latex
        \theoremstyle{plain}
        \newtheorem{thm}[subsection]{Theorem}
        \newtheorem{prop}[subsection]{Proposition}
        \newtheorem{cor}[subsection]{Corollary}
        \newtheorem{remark}{Remark}
        \newtheorem{conj}[subsection]{Conjecture}
        \newtheorem*{conj*}{Conjecture}
         ```

        defines the `thm`, `prop`, `cor`, and `conj` environments to be numbered using the `subsection` counter, the `remark` environmment to be defiend as an unnumbered environment, and the `conj*` environment to be defined as an unnumbered environment with a different name than the `conj` environment.

    - The `\swapnumbers` command is included in the preamble to change the way that theorems are numbered in the document, e.g. the article has `1.2 Theorem` as opposed to `Theorem 1.2`.
    - The equations are numbered along the subsections - this is accomplished by the lines 

        ```latex
        \numberwithin{equation}{subsection}
        \renewcommand{\theequation}{\thesubsection.\arabic{equation}}
        ```

        in the preamble.
- Hoyois, *[A quadratic refinement of the Grothendieck-Lefschetz-Verdier Trace Formula](https://arxiv.org/abs/1309.6147)*
    - The theorem-like environments are numbered `Theorem 1.1, Theorem 1.3, Corollary 1.4, Theorem 1.5`, etc.
        - The theorem-like environments that are numbered are assigned the `equation` counter. In particular, the equation
        environments share their numberings with the theorem-like environments. For example, section 1 has Equation `(1.2)`
        - This equation counter is reset at the beginning of each section and the section number is included in the numbering via
        ```latex 
        \numberwithin{equation}{section}
        ```

In [7719]:
# TODO: consider different arxiv articles to see how they are numbered

In [7720]:
#| export
def _search_counters_by_pattern(
        preamble: str,
        newtheorem_regex: re.Pattern, # This is supposed to be a regex that detects and captures parameters of `\newtheorem` commands.
        counter_group: int # This depends on which `newtheorem_regex` is used, and is either 3 or 4. 
        ) -> dict[str, str]: # The 
    """
    Capture the newly defined theorem-like environment names as well as the
    counters that they belong to
    
    This is a helper function for `numbered_newtheorems_counters_in_preamble`.
    
    """
    counters = {}
    for match in newtheorem_regex.finditer(preamble):
        env_name = match.group(1)
        counter = match.group(counter_group)
        # If no counter was specified, use the environment name as the counter
        if counter is None:
            counter = env_name
        counters[env_name] = counter
    return counters

In [7721]:
#| hide

# Test that the contents of the `counters_for_environments` function are detecting
# The defined commands correctly.
text = text_from_file(_test_directory() / 'latex_examples' / 'newtheorem_example.tex') 
preamble, _ = divide_preamble(text)

second_results = _search_counters_by_pattern(preamble, SECOND_PARAMETER_PATTERN, 3)
third_results = _search_counters_by_pattern(preamble, THIRD_PARAMETER_PATTERN, 4)
assert 'remark' not in second_results
assert 'remark' in third_results

In [7722]:
#| hide
preamble = text = r"""
\theoremstyle{plain}
\newtheorem{thm}[subsection]{Theorem}
\newtheorem{prop}[subsection]{Proposition}
\newtheorem{cor}[subsection]{Corollary}
\newtheorem{remark}{Remark}
\newtheorem{conj}[subsection]{Conjecture}
\newtheorem*{conj*}{Conjecture}
"""

second_results = _search_counters_by_pattern(preamble, SECOND_PARAMETER_PATTERN, 3)
third_results = _search_counters_by_pattern(preamble, THIRD_PARAMETER_PATTERN, 4)

second_results
# third_results

{'thm': 'subsection',
 'prop': 'subsection',
 'cor': 'subsection',
 'remark': 'remark',
 'conj': 'subsection'}

In [7723]:
#| export
def _article_is_amsart_or_article(
        preamble: str # The preamble with no comments
        ):
    """
    helper function of `numbered_newtheorems_counters_in_preamble`.
    """
    return bool(re.search(r'\\documentclass\s*(\[\s*(.*?)\s*\])?\s*\{\s*(amsart|article)\}', preamble))

In [7724]:
#| hide
assert _article_is_amsart_or_article(r'\documentclass[12pt,letterpaper]{amsart}')
assert _article_is_amsart_or_article(r'\documentclass{  amsart}')
assert _article_is_amsart_or_article(r'\documentclass{amsart}')
# When I tried compiling a sample LaTeX document, I found that having spaces before, but not after, `amsart` is fine.
assert not _article_is_amsart_or_article(r'\documentclass{  amsart }')  
assert not _article_is_amsart_or_article(r'\documentclass[12pt,letterpaper]{art}')

assert _article_is_amsart_or_article(
    r'''\documentclass{amsart}
    Lorem Ipsum''')
    
assert _article_is_amsart_or_article(r'\documentclass[12pt,letterpaper]{article}')
assert _article_is_amsart_or_article(r'\documentclass{  article}')
assert _article_is_amsart_or_article(r'\documentclass{article}')
# When I tried compiling a sample LaTeX document, I found that having spaces before, but not after, `amsart` is fine.
assert not _article_is_amsart_or_article(r'\documentclass{  article }')  
assert not _article_is_amsart_or_article(r'\documentclass[12pt,letterpaper]{art}')

assert _article_is_amsart_or_article(
    r'''\documentclass{article}
    Lorem Ipsum''')

In [7725]:
#| export
def _combine_second_and_third_paramter_results(preamble):
    """
    Inspect invocations of the `\newtheorem` command in the preamble,
    separately dealing with invocations with a third optional parameter vs.
    a second optional parameter.

    helper function of `numbered_newtheorems_counters_in_preamble`.
    """
    second_results = _search_counters_by_pattern(preamble, SECOND_PARAMETER_PATTERN, 3)
    third_results = _search_counters_by_pattern(preamble, THIRD_PARAMETER_PATTERN, 4)
    to_return = {}
    for environment_name, counter in second_results.items():
        to_return[environment_name] = (counter, None)
    for environment_name, reset_counter in third_results.items():
        if environment_name in to_return:
            continue
        to_return[environment_name] = (environment_name, reset_counter)
    return to_return        

In [7726]:
#| export
def numbered_newtheorems_counters_in_preamble(
        document: str, # The LaTeX document
        add_equation_counter: Optional[bool] = None, # Determines whether or not the `equation` environment will have a counter added when a `newthoerem` command for the `equation` environment is not explicitly invoked in the preamble. If `None`, then the counter is added if the article is of class `amsart` or `article`. If `True`, then the counter is added. If `False`, then the counter is not added.
        ) -> dict[str, tuple[str, Union[str, None]]]: # The keys are the command names of the environments. The value a key is a tuple `(<counter>, <reset_by_counter>)`, where `<counter>` is the counter that the environment belongs to, which can be custom defined or predefined in LaTeX, and `<reset_by_counter>` is a counter whose incrementation resets the # counter of the environment, if available. 
    r"""Return the dict specifying the numbered `\newtheorem` command invocations

    Assumes that

    - invocations of the `\newtheorem` command are exclusively in the
    preamble of the LaTeX document.
    - theorem-like environments are defined using the `\newtheorem` command.
    - no environments of the same name are defined twice.
    - There is at most one invocation of `\theoremstyle` or `\newtheorem` in each line.

    This function does not take into account `\numberwithins` being used.
    The `numberwithins_in_preamble` function accounts for invocations of
    the `\numberwithins` command instead.

    The `equation` environment (and other related environments, such as `eqnarray`)
    seems to be included in documents of
    the class `amsart` or `article` (i.e. documents which invoke
    `\documentclass{amsart}` or `\documentclass{article}`,
    possibly with some optional arguments).
    The `equation` environment (and other related environments) is accordingly included
    in the output
    of this function if the document is of the class `amsart` and 
    `add_equation_counter` is not specified, set to `None`.

    This function uses two separate regex patterns, one to detect the invocations
    of `\newtheorem` in which the optional parameter is the second parameter and
    one to detect those in which the optional parameter is the third parameter.


    """
    preamble, _ = divide_preamble(document)
    preamble = remove_comments(preamble)
    # TODO: maybe use the `regex` package instead of `re` with a recursive
    # balanced-curly braces detecting regex.
    commands_and_counters = _combine_second_and_third_paramter_results(preamble)
    # --- ADD THIS LOGIC ---
    # Map manual environments that use the equation counter
    manual_envs = ['remark', 'example', 'heuristic', 'warning']
    for env in manual_envs:
    # Use a regex to see if the environment is actually used in a \begin{} 
    # or if the environment name appears in the cleaned preamble
        if env not in commands_and_counters:
            # Check if \begin{env} exists in the document or if it's defined via \newenvironment
            if regex.search(rf'\\begin\s*\{{{env}\}}|\\newenvironment\s*\{{{env}\}}', document):
                commands_and_counters[env] = ('equation', None)
    # for env in manual_envs:

    #     if env in document and env not in commands_and_counters:
    #          commands_and_counters[env] = ('equation', None)
    # ----------------------
    if 'equation' not in commands_and_counters and (
            add_equation_counter == True or
            add_equation_counter is None and _article_is_amsart_or_article(preamble)):
        commands_and_counters['equation'] = ('equation', None)  
        if 'eqnarray' not in commands_and_counters:
            commands_and_counters['eqnarray'] = ('equation', None)

    return commands_and_counters



In [7727]:
#| hide
def test_manual_environment_mapping():
    # Wrap the environment in a proper LaTeX document structure
    latex_text = r"""
    \documentclass{article}
    \newenvironment{remark}[1][]{
       \refstepcounter{equation}
       Remark content...
    }
    \begin{document}
    % Usage of the environment to trigger the regex search
    \begin{remark}
    This is a remark.
    \end{remark}
    \end{document}
    """
    # We pass True to add_equation_counter to ensure equation is tracked
    counters = numbered_newtheorems_counters_in_preamble(latex_text, add_equation_counter=True)
    
    # Remark should now be mapped to the equation counter
    test_eq(counters['remark'], ('equation', None))
    # Equation itself should still be there
    test_eq(counters['equation'], ('equation', None))

test_manual_environment_mapping()

In [7728]:
text = r"""\theoremstyle{definition}                 \newtheorem{conj}{Conjecture}
\newtheorem*{example}{Example}            \newtheorem{defn}{Definition}
\newtheorem{remark}{Remark} \newtheorem*{notation}{Notation}
\begin{document}
\end{document}"""
numbered_newtheorems_counters_in_preamble(text)

{'conj': ('conj', None), 'defn': ('defn', None), 'remark': ('remark', None)}

The `numbered_newtheorems_counter_in_preamble` function parses the preamble of a LaTeX document for invocations of the `\newtheorem` command and returns what counters each theorem-like environment command belongs to.

In [7729]:
text = text_from_file(_test_directory() / 'latex_examples' / 'newtheorem_example.tex') 
print(text)

counters = numbered_newtheorems_counters_in_preamble(text)
test_eq(counters,
   {'theorem': ('theorem', None), 'lemma': ('theorem', None), 'definition': ('theorem', None), 'corollary': ('corollary', None), 'remark': ('remark', 'theorem'), 'equation': ('equation', None), 'eqnarray': ('equation', None)}
)

\documentclass{article}
\usepackage{amsthm}

\newtheorem{theorem}{Theorem}
\newtheorem{lemma}[theorem]{Lemma}
\newtheorem{definition}[theorem]{Definition} % Note that `theorem`, `lemma`, and `definition` all have `theorem` as their counter.
\newtheorem{corollary}{Corollary} % Note that `corollary` has its own counter.
\newtheorem{remark}{Remark}[theorem] % `remark` has `theorem` as its counter
\newtheorem*{conjecture*}{Conjecture} % `conjecture*` has no counter

\begin{document}

\section{Introduction}

\begin{theorem}
This is Theorem 1.
\end{theorem}

\begin{lemma}
This is Lemma 2.
\end{lemma}

\begin{definition}
This is Definition 3.
\end{definition}

\end{document}


In [7730]:
text = r"""
\theoremstyle{plain}
\newtheorem{thm}[subsection]{Theorem}
\newtheorem{prop}[subsection]{Proposition}
\newtheorem{cor}[subsection]{Corollary}
\newtheorem{remark}{Remark}
\newtheorem{conj}[subsection]{Conjecture}
\newtheorem*{conj*}{Conjecture}
\begin{document}
\end{document}
"""
counters = numbered_newtheorems_counters_in_preamble(text)
test_eq(
    counters,
    {'thm': ('subsection', None), 'prop': ('subsection', None), 'cor': ('subsection', None), 'remark': ('remark', None), 'conj': ('subsection', None)})

`numbered_newtheorems_counters_in_preamble` ignores commented out text:

In [7731]:
text = r"""
\theoremstyle{plain}
\newtheorem{thm}[subsection]{Theorem}
\newtheorem{prop}[subsection]{Proposition}
\newtheorem{cor}[subsection]{Corollary}
% \newtheorem{remark}{Remark}
\newtheorem{conj}[subsection]{Conjecture}
\newtheorem*{conj*}{Conjecture} %\newtheorem{fakeenv}{This won't be picked up!}
\begin{document}
\end{document}
"""
counters = numbered_newtheorems_counters_in_preamble(text)
test_eq(
    counters,
    {'thm': ('subsection', None), 'prop': ('subsection', None), 'cor': ('subsection', None), 'conj': ('subsection', None)})

`numbered_newtheorems_counters_in_preamble` does not account for `\numberwithin` command invocations. The `numberwithins_in_preamble` function accounts for invocations of `\numberwithin` instead.

In [7732]:
text = text_from_file(_test_directory() / 'latex_examples' / 'numbering_example_3_theorem_like_environments_share_counter_with_equation_and_reset_at_each_section' / 'main.tex')
print(text)
# So `numbered_newtheorems_counters_in_preamble` only considers
# the theorem-like environemnts as being counted by 'equation'.
# Note that the command  `\numberwithin{equation}{section}`
# resets the equation counter
# every time the `section` counter is incremented.
test_eq(numbered_newtheorems_counters_in_preamble(text), 
       {'theorem': ('equation', None), 'proposition': ('equation', None), 'lemma': ('equation', None), 'corollary': ('equation', None), 'definition': ('equation', None), 'example': ('equation', None), 'remark': ('equation', None), 'equation': ('equation', None), 'eqnarray': ('equation', None)}
        )

\documentclass{amsart}
\usepackage[utf8]{inputenc}
\usepackage{amsmath, amsfonts, amssymb, amsthm, amsopn}

\numberwithin{equation}{section}

\theoremstyle{plain}
\newtheorem*{theorem*}{Theorem}
\newtheorem*{theoremA}{Theorem A}
\newtheorem*{theoremB}{Theorem B}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem{proposition}[equation]{Proposition}
\newtheorem{lemma}[equation]{Lemma}
\newtheorem{corollary}[equation]{Corollary}

\theoremstyle{definition}
\newtheorem{definition}[equation]{Definition}
\newtheorem{example}[equation]{Example}
\newtheorem*{acknowledgements}{Acknowledgements}
\newtheorem*{conventions}{Conventions}

\theoremstyle{remark}
\newtheorem{remark}[equation]{Remark}

\begin{document}

\section{Introduction}

\begin{theorem}
This is Theorem 1.1. This is because the \verb|\numberwithin{equation}{section}| makes the section number included in the equation counter and because the \\
\verb|\newtheorem{theorem}[equation]{Theorem}| command makes the environment \verb|theorem

The `\newtheorem` command can be used to specify the counter of the newly defined theorem-like environment to be reset upon another counter's incrementation; for example `\newtheorem{theorem}{Theorem}[section]` specifies for a new environment named `theorem` (with display text `Theorem`) that is reset whenever the `section` counter is incremented.

In [7733]:
text = text_from_file(_test_directory() / 'latex_examples' / 'numbering_example_7_newtheorem_command_restarts_counter_by_section' / 'main.tex') 
print(text)
# So `numbered_newtheorems_counters_in_preamble` only considers the theorem-like
#  environemnts as being counted by 'equation'.
# Note that the command  `\numberwithin{equation}{section}` resets the equation counter
# every time the `section` counter is incremented.

test_eq(numbered_newtheorems_counters_in_preamble(text), 
        {'lemma': ('theorem', None), 'theorem': ('theorem', 'section'), 'corollary': ('corollary', 'theorem'), 'proposition': ('proposition', 'section'), 'equation': ('equation', None), 'eqnarray': ('equation', None)}

        )


% Based on an example from https://www.overleaf.com/learn/latex/Theorems_and_proofs#Numbered_theorems.2C_definitions.2C_corollaries_and_lemmas

\documentclass[12 pt]{amsart}

\newtheorem{theorem}{Theorem}[section]
\newtheorem{corollary}{Corollary}[theorem]
\newtheorem{lemma}[theorem]{Lemma}
% Note that the below invocation of \newtheorem is invalid:
% \newtheorem{proposition}[theorem]{Proposition}[section]
\newtheorem{proposition}{Proposition}[section]

\begin{document}
\section{Introduction}
Theorems can easily be defined:

\begin{theorem}
Let \(f\) be a function whose derivative exists in every point, then \(f\) is 
a continuous function.
\end{theorem}

\begin{theorem}[Pythagorean theorem]
\label{pythagorean}
This is a theorem about right triangles and can be summarised in the next 
equation 
\[ x^2 + y^2 = z^2 \]
\end{theorem}

And a consequence of theorem \ref{pythagorean} is the statement in the next 
corollary.

\begin{corollary}
There's no right rectangle whose sides measure 3c

In [7734]:
#| hide
# TODO
# I found a bug where the section numbering cannot handle the theorem-like environment defined like
# \newtheorem{theorem}{Theorem}[section], cf. https://arxiv.org/abs/2106.10586 and the example in


For the following test, we have multiple theorems defined in the same line:

In [7735]:
text = r"""\theoremstyle{definition}                 \newtheorem{conj}{Conjecture}
\newtheorem*{example}{Example}            \newtheorem{defn}{Definition}
\newtheorem{remark}{Remark} \newtheorem*{notation}{Notation}
\begin{document}
\end{document}"""
numbered_newtheorems_counters_in_preamble(text)

{'conj': ('conj', None), 'defn': ('defn', None), 'remark': ('remark', None)}

In [7736]:
#| export
def numberwithins_in_preamble(
        document: str # The LaTeX document
    ) -> dict[str, str]: # The keys are the first arguments of `numberwithin` invocations and the values ar ethe second arguments of `numberwithin` invocations.
    r"""Return the `dict` describing `\numberwithin` commands invoked
    in the preamble of `document`.
    
    Assumes that `\numberwithin` commands are invoked exclusively in the
    preamble.

    See also the `numbered_newtheorems_counter_in_preamble` function,
    which parses invocations of the `\newtheorem` command.
    """
    preamble, _ = divide_preamble(document)
    preamble = remove_comments(preamble)
    pattern = regex.compile(r'\\numberwithin\s*\{\s*(\w+)\s*\}\s*\{\s*(.*)\s*\}')
    numberwithins = {}

    for match in pattern.finditer(preamble):
        environment_to_number = match.group(1)
        environment_to_count = match.group(2)
        numberwithins[environment_to_number] = environment_to_count

    return numberwithins

The `numberwithins_in_preamble` function returns a `dict` describing invocations of the `\numberwithin` commands. See also the `numbered_newtheorems_counter_in_preamble` function, which parses invocations of the `\newtheorem` command.

In the following example, there is an invocation of the `\numberwithin` command; for the LaTeX document in the example below, the equation counter is reset every time the `section` counter is incremented. 

The `numberwithins_in_preamble` function returns a `dict` that is used by the `divide_latex_text` function to account for this fact.

In [7737]:
text = text_from_file(_test_directory() / 'latex_examples' / 'numbering_example_3_theorem_like_environments_share_counter_with_equation_and_reset_at_each_section' / 'main.tex')
print(text)
test_eq(numberwithins_in_preamble(text), {'equation': 'section'})

\documentclass{amsart}
\usepackage[utf8]{inputenc}
\usepackage{amsmath, amsfonts, amssymb, amsthm, amsopn}

\numberwithin{equation}{section}

\theoremstyle{plain}
\newtheorem*{theorem*}{Theorem}
\newtheorem*{theoremA}{Theorem A}
\newtheorem*{theoremB}{Theorem B}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem{proposition}[equation]{Proposition}
\newtheorem{lemma}[equation]{Lemma}
\newtheorem{corollary}[equation]{Corollary}

\theoremstyle{definition}
\newtheorem{definition}[equation]{Definition}
\newtheorem{example}[equation]{Example}
\newtheorem*{acknowledgements}{Acknowledgements}
\newtheorem*{conventions}{Conventions}

\theoremstyle{remark}
\newtheorem{remark}[equation]{Remark}

\begin{document}

\section{Introduction}

\begin{theorem}
This is Theorem 1.1. This is because the \verb|\numberwithin{equation}{section}| makes the section number included in the equation counter and because the \\
\verb|\newtheorem{theorem}[equation]{Theorem}| command makes the environment \verb|theorem

In [7738]:
#| export
# def get_counter_hierarchy(document: str) -> dict[str, str]:
#     """Builds a map of child_counter -> parent_counter."""
#     # 1. Get initial resets from \newtheorem (the [section] or [subsection] part)
#     thm_counters = numbered_newtheorems_counters_in_preamble(document)
#     # 2. Get overrides from \numberwithin
#     explicit_resets = numberwithins_in_preamble(document)

#     # --- ADD THIS LOGIC ---
#     # Manually search for \@addtoreset{equation}{section}
#     addtoreset_pattern = regex.compile(r'\\@addtoreset\s*\{\s*(\w+)\s*\}\s*\{\s*(\w+)\s*\}')
#     preamble, _ = divide_preamble(document)
#     for match in addtoreset_pattern.finditer(preamble):
#         explicit_resets[match.group(1)] = match.group(2)
#     # ----------------------
    
#     # Start with your built-in LaTeX defaults
#     hierarchy = {
#         'subsection': 'section',
#         'subsubsection': 'subsection',
#         'paragraph': 'subsubsection'
#     }

#     # Add resets found in \newtheorem{thm}{Theorem}[section]
#     for env_name, (counter, reset_by) in thm_counters.items():
#         if reset_by:
#             hierarchy[counter] = reset_by
            
#     # Add/Override with explicit \numberwithin{thm}{subsection}
#     # This is what fixes your specific preamble issue
#     hierarchy.update(explicit_resets)
    
#     return hierarchy

In [7739]:
#| export
def get_counter_hierarchy(document: str) -> dict[str, str]:
    """Builds a map of child_counter -> parent_counter."""
    thm_counters = numbered_newtheorems_counters_in_preamble(document)
    explicit_resets = numberwithins_in_preamble(document)

    preamble, _ = divide_preamble(document)
    addtoreset_pattern = regex.compile(r'\\@addtoreset\s*\{\s*(\w+)\s*\}\s*\{\s*(\w+)\s*\}')
    for match in addtoreset_pattern.finditer(preamble):
        explicit_resets[match.group(1)] = match.group(2)
    
    # Start with standard defaults
    hierarchy = {
        'section': 'part',         # <--- ADD THIS: section is reset by part
        'subsection': 'section',
        'subsubsection': 'subsection',
        'paragraph': 'subsubsection'
    }

    # --- ADD THIS LINE ---
    # Ensure equation climbs to section by default if not otherwise specified
    if 'equation' not in explicit_resets:
        hierarchy['equation'] = 'section'

    for env_name, (counter, reset_by) in thm_counters.items():
        if reset_by:
            hierarchy[counter] = reset_by
            
    hierarchy.update(explicit_resets)
    return hierarchy

In [7740]:
#| hide

def test_get_counter_hierarchy_new():
    # Case 1: Standard Article Defaults
    # Should include built-ins: subsection -> section
    # Should include our new default: equation -> section
    latex_basic = r"""\documentclass{article}\begin{document}\end{document}"""
    h_basic = get_counter_hierarchy(latex_basic)
    test_eq(h_basic.get('subsection'), 'section')
    test_eq(h_basic.get('equation'), 'section')

    # Case 2: Manual @addtoreset override
    # If the user explicitly wants equations to reset by subsection
    latex_addtoreset = r"""
    \documentclass{article}
    \makeatletter
    \@addtoreset{equation}{subsection}
    \makeatother
    \begin{document}\end{document}
    """
    h_reset = get_counter_hierarchy(latex_addtoreset)
    test_eq(h_reset.get('equation'), 'subsection')

    # Case 3: \numberwithin override
    # This should take priority over our 'section' default
    latex_numberwithin = r"""
    \documentclass{article}
    \numberwithin{equation}{subsection}
    \begin{document}\end{document}
    """
    h_nw = get_counter_hierarchy(latex_numberwithin)
    test_eq(h_nw.get('equation'), 'subsection')

    # Case 4: Theorem-specific resets from \newtheorem
    # \newtheorem{thm}{Theorem}[section]
    latex_thm = r"""
    \documentclass{article}
    \newtheorem{thm}{Theorem}[section]
    \begin{document}\end{document}
    """
    h_thm = get_counter_hierarchy(latex_thm)
    # The counter 'thm' is reset by 'section'
    test_eq(h_thm.get('thm'), 'section')

    # Case 5: Complex Chain
    # equation -> subsection -> section
    latex_complex = r"""
    \documentclass{article}
    \numberwithin{equation}{subsection}
    \begin{document}\end{document}
    """
    h_complex = get_counter_hierarchy(latex_complex)
    test_eq(h_complex.get('equation'), 'subsection')
    test_eq(h_complex.get('subsection'), 'section')

test_get_counter_hierarchy_new()

In [7741]:
#| hide
def test_addtoreset_detection():
    # We must include \begin{document} so divide_preamble can function
    latex_text = r"""
    \documentclass{article}
    \makeatletter
    \@addtoreset{equation}{section}
    \makeatother
    \newtheorem{thm}{Theorem}[section]
    \begin{document}
    Content here.
    \end{document}
    """
    hierarchy = get_counter_hierarchy(latex_text)
    
    # Check if equation was correctly linked to section
    test_eq(hierarchy.get('equation'), 'section')
    # Verify built-ins still exist
    test_eq(hierarchy.get('subsection'), 'section')

test_addtoreset_detection()

In [7742]:
#| export
def get_reset_hierarchy(document: str) -> dict[str, list[str]]:
    """Builds a map of parent_counter -> list_of_child_counters for resetting."""
    upward_hierarchy = get_counter_hierarchy(document)
    reset_hierarchy = {}
    for child, parent in upward_hierarchy.items():
        if parent not in reset_hierarchy:
            reset_hierarchy[parent] = []
        if child not in reset_hierarchy[parent]:
            reset_hierarchy[parent].append(child)
    return reset_hierarchy

In [7743]:
#| hide
#| hide

def test_get_reset_hierarchy():
    # Case 1: Simple hierarchy from a string
    latex_1 = r"""
    \documentclass{article}
    \newtheorem{theorem}{Theorem}[section]
    \numberwithin{equation}{section}
    \begin{document}
    \end{document}
    """
    resets = get_reset_hierarchy(latex_1)
    assert 'section' in resets
    # Note: 'equation' is here both by default and explicit numberwithin
    test_eq(set(resets['section']), {'theorem', 'equation', 'subsection'})
    
    # Case 2: Deep nesting (The "2.1.1" chain)
    latex_2 = r"""
    \documentclass{article}
    \newtheorem{theorem}{Theorem}[subsection]
    \begin{document}
    \end{document}
    """
    resets_deep = get_reset_hierarchy(latex_2)
    
    # UPDATED EXPECTATION: section resets subsection AND the default equation
    test_eq(set(resets_deep['section']), {'subsection', 'equation'})
    
    # subsection resets theorem (from preamble) AND subsubsection (from built-in defaults)
    test_eq(set(resets_deep['subsection']), {'theorem', 'subsubsection'})
    
    # Case 3: Verify no infinite loops or duplicates
    latex_3 = r"""
    \documentclass{article}
    \newtheorem{thm}{Theorem}[section]
    \newtheorem{prop}[thm]{Proposition}
    \newtheorem{lem}[thm]{Lemma}
    \begin{document}
    \end{document}
    """
    resets_shared = get_reset_hierarchy(latex_3)
    # Check that both the default 'equation' and our custom 'thm' are present
    test_eq(set(resets_shared['section']), {'thm', 'equation', 'subsection'})

test_get_reset_hierarchy()


In [7744]:

# --- Integration Test with _recursive_reset ---

def test_hierarchy_integration_reset():
    # We use a preamble that explicitly defines the hierarchy
    latex = r"""
    \documentclass{article}
    \newtheorem{theorem}{Theorem}[subsection]
    \begin{document}
    \end{document}
    """
    # 1. Get the hierarchy for resetting
    # This should yield {'section': ['subsection'], 'subsection': ['theorem', 'subsubsection']}
    all_withins = get_reset_hierarchy(latex)
    
    # Debug: Print this to be 100% sure the mapping exists
    # print(f"DEBUG: Reset map is {all_withins}")
    
    # 2. Setup a "dirty" state
    counters = {'section': 1, 'subsection': 4, 'theorem': 12, 'subsubsection': 5}
    
    # 3. Reset the section
    _recursive_reset('section', counters, all_withins)
    
    # 4. Check the cascade
    test_eq(counters['section'], 1)       # Parent preserved
    test_eq(counters['subsection'], 0)    # Child reset (The 4 -> 0 fix)
    test_eq(counters['theorem'], 0)       # Grandchild reset (The 12 -> 0 fix)
    test_eq(counters['subsubsection'], 0) # Built-in reset

test_hierarchy_integration_reset()

In [7745]:

def test_hierarchy_logic():
    # Example 14 context: part -> section -> remark/equation
    doc = r"""
    \numberwithin{equation}{section}
    \begin{document}
    \part{Algebra}
    \section{Modules}
    \end{document}
    """
    hierarchy = get_counter_hierarchy(doc)
    resets = get_reset_hierarchy(doc)
    
    print(f"Hierarchy: {hierarchy}")
    print(f"Resets: {resets}")
    
    # In Example 14:
    # section should have part as parent
    # equation (or remark) should have section as parent
    assert hierarchy.get('section') == 'part'
    assert 'section' in resets.get('part', [])
    assert 'equation' in resets.get('section', [])

test_hierarchy_logic()


Hierarchy: {'section': 'part', 'subsection': 'section', 'subsubsection': 'subsection', 'paragraph': 'subsubsection', 'equation': 'section'}
Resets: {'part': ['section'], 'section': ['subsection', 'equation'], 'subsection': ['subsubsection'], 'subsubsection': ['paragraph']}


In [7746]:
#| export
# def get_formatted_number(counter_name: str, current_counts: dict, hierarchy: dict) -> str:
#     """Recursively builds the number string (e.g., '1.2.1')"""
#     numbers = [str(current_counts.get(counter_name, 0))]
    
#     cursor = counter_name
#     while cursor in hierarchy:
#         parent = hierarchy[cursor]
#         numbers.append(str(current_counts.get(parent, 0)))
#         cursor = parent
        
#     return ".".join(reversed(numbers))

def get_formatted_number(
        env_or_counter: str, 
        counters: dict[str, int], 
        numberwithins: dict[str, str], 
        numbertheorem_counters: dict[str, tuple[str, Union[str, None]]]) -> str:
    
    if env_or_counter in numbertheorem_counters:
        counter_name, _ = numbertheorem_counters[env_or_counter]
    else:
        counter_name = env_or_counter

    current_val = counters.get(counter_name, 0)
    parent = numberwithins.get(counter_name)

    # CRITICAL: Only recurse if the parent exists AND its value is > 0.
    if parent and parent in counters and counters[parent] > 0:
        parent_prefix = get_formatted_number(
            parent, counters, numberwithins, numbertheorem_counters)
        return f"{parent_prefix}.{current_val}"
    
    return str(current_val)

In [7747]:
#| hide
def test_zero_parent_omission():
    # Scenario: section is within part, but part hasn't been used (value is 0)
    counters = {'part': 0, 'section': 1, 'equation': 1}
    numberwithins = {'section': 'part', 'equation': 'section'}
    
    # Section should be "1", not "0.1"
    section_num = get_formatted_number('section', counters, numberwithins, {})
    print(f"Section Number (Expected '1'): {section_num}")
    assert section_num == "1"
    
    # Equation should be "1.1", not "0.1.1"
    eq_num = get_formatted_number('equation', counters, numberwithins, {})
    print(f"Equation Number (Expected '1.1'): {eq_num}")
    assert eq_num == "1.1"

    # Scenario: Part IS used (value is 1)
    counters_with_part = {'part': 1, 'section': 1, 'equation': 1}
    eq_num_with_part = get_formatted_number('equation', counters_with_part, numberwithins, {})
    print(f"Equation Number with Part (Expected '1.1.1'): {eq_num_with_part}")
    assert eq_num_with_part == "1.1.1"

test_zero_parent_omission()

Section Number (Expected '1'): 1
Equation Number (Expected '1.1'): 1.1
Equation Number with Part (Expected '1.1.1'): 1.1.1


In [7748]:
#| hide
def test_ghost_parent():
    # Example 1 state: No parts, just sections.
    counters = {'section': 1}
    numberwithins = {} # No parent for section
    
    # If this returns "0.1", Example 1 will fail. 
    # It MUST return "1".
    num_str = get_formatted_number('section', counters, numberwithins, {})
    print(f"Formatted Section Number: '{num_str}'")
    
    assert num_str == "1"

test_ghost_parent()

Formatted Section Number: '1'


In [7749]:
#| hide
#| hide
def test_full_hierarchical_chain():
    # Setup state
    hierarchy = {
        'equation': 'section',
        'subsection': 'section'
    }
    
    # Mock nthm_counters to show how 'thm' maps to 'equation'
    nthm_counters = {
        'thm': ('equation', None),
        'equation': ('equation', None),
        'subsection': ('subsection', None)
    }
    
    # Current state: Section 2, Equation 1
    current_counts = {
        'section': 2,
        'equation': 1
    }
    
    # 1. Test using the environment name 'thm'
    # It should: map 'thm' -> 'equation' -> find parent 'section' -> '2.1'
    formatted_num = get_formatted_number('thm', current_counts, hierarchy, nthm_counters)
    test_eq(formatted_num, '2.1')
    
    # 2. Test for a Subsection
    # It should: map 'subsection' -> 'subsection' -> find parent 'section' -> '2.3'
    current_counts['subsection'] = 3
    formatted_sub = get_formatted_number('subsection', current_counts, hierarchy, nthm_counters)
    test_eq(formatted_sub, '2.3')

test_full_hierarchical_chain()

In [7750]:
#| hide
# --- Updated Test Data Setup ---

# We need the nthm_counters from the preamble to handle the mapping
nthm_counters = numbered_newtheorems_counters_in_preamble(sample_preamble)
hierarchy = get_counter_hierarchy(sample_preamble)

# Verification of current state
test_eq(hierarchy['theorem'], 'subsection')
test_eq(hierarchy['subsection'], 'section')

mock_counts = {
    'section': 1,
    'subsection': 2,
    'theorem': 3,
    'equation': 5
}

# 1. Test the full 1.2.3 chain - Pass nthm_counters!
test_eq(get_formatted_number('theorem', mock_counts, hierarchy, nthm_counters), "1.2.3")

# 2. Test a 1.2.5 equation
test_eq(get_formatted_number('equation', mock_counts, hierarchy, nthm_counters), "1.2.5")

# 3. Test a simple 1.2 subsection
test_eq(get_formatted_number('subsection', mock_counts, hierarchy, nthm_counters), "1.2")

# 4. Test complex chains (2.1.4.1)
complex_hierarchy = {
    'subthm': 'theorem',
    'theorem': 'subsection',
    'subsection': 'section'
}
# Mocking nthm_counters for the complex chain
complex_nthm = {
    'subthm': ('subthm', None),
    'theorem': ('theorem', None),
    'subsection': ('subsection', None)
}
complex_counts = {'section': 2, 'subsection': 1, 'theorem': 4, 'subthm': 1}
test_eq(get_formatted_number('subthm', complex_counts, complex_hierarchy, complex_nthm), "2.1.4.1")

In [7751]:
#| hide
def test_title_formatting():
    # Mocking a state where section 1 is inside part 1
    counters = {'part': 1, 'section': 1, 'equation': 1}
    numberwithins = {'section': 'part', 'equation': 'section'}
    
    # For a Section title: In standard LaTeX, \section{...} shows "1" not "1.1" 
    # UNLESS it's a report/book class. 
    # We need to see what your code produces:
    sec_num = get_formatted_number('section', counters, numberwithins, {})
    rem_num = get_formatted_number('equation', counters, numberwithins, {'remark': ('equation', 'section')})
    
    print(f"Section Number String: {sec_num}")
    print(f"Remark Number String: {rem_num}")
    
    # This will identify if your 'get_formatted_number' is too aggressive 
    # for top-level sectioning.

#### Getting the display names of environment

For example, `\newtheorem{theorem}{Theorem}` defines a theorem-like environment called `theorem` whose display name is `Theorem`.

In [7752]:
#| export
def _search_display_names_by_pattern(
        preamble: str,
        newtheorem_regex: re.Pattern,
        display_name_group: int # This depends on which `newtheorem_regex` is used, and is either 3 or 4. 
        ) -> dict[str, str]:
    """
    Capture the newly defined theorem-like environment names as well as the
    counters that they belong to"""
    display_names = {}
    for match in newtheorem_regex.finditer(preamble):
        env_name = match.group(1)
        display_name = match.group(display_name_group)
        display_names[env_name] = display_name
    return display_names

In [7753]:
#| export
def display_names_of_environments(
        document: str # The LaTeX document
        ) -> dict[str, str]:  
    r"""Return the dict specifying the display names for each theorem-like
    environment.

    This function uses two separate regex patterns, one to detect the invocations
    of `\newtheorem`
    in which the optional parameter is the second parameter and one to detect
    those in which the optional parameter is the third parameter.

    Assumes that
    - invocations of the `\newtheorem` command are exclusively in the
    preamble of the LaTeX document.
    - theorem-like environments are defined using the `\newtheorem` command.
    - no environments of the same name are defined twice.

    """
    preamble, _ = divide_preamble(document)
    second_results = _search_display_names_by_pattern(preamble, SECOND_PARAMETER_PATTERN_WITH_OPTIONAL_STAR, 4)
    third_results = _search_display_names_by_pattern(preamble, THIRD_PARAMETER_PATTERN_WITH_OPTIONAL_STAR, 2)
    return second_results | third_results
    


Basic examples:

In [7754]:
text = text_from_file(_test_directory() / 'latex_examples' / 'newtheorem_example.tex') 
print(text)
display_names = display_names_of_environments(text)
test_eq(display_names,{'theorem': 'Theorem', 'lemma': 'Lemma', 'definition': 'Definition', 'corollary': 'Corollary', 'conjecture*': 'Conjecture', 'remark': 'Remark'})

\documentclass{article}
\usepackage{amsthm}

\newtheorem{theorem}{Theorem}
\newtheorem{lemma}[theorem]{Lemma}
\newtheorem{definition}[theorem]{Definition} % Note that `theorem`, `lemma`, and `definition` all have `theorem` as their counter.
\newtheorem{corollary}{Corollary} % Note that `corollary` has its own counter.
\newtheorem{remark}{Remark}[theorem] % `remark` has `theorem` as its counter
\newtheorem*{conjecture*}{Conjecture} % `conjecture*` has no counter

\begin{document}

\section{Introduction}

\begin{theorem}
This is Theorem 1.
\end{theorem}

\begin{lemma}
This is Lemma 2.
\end{lemma}

\begin{definition}
This is Definition 3.
\end{definition}

\end{document}


In [7755]:
file = _test_directory() / 'latex_examples' / 'numbering_example_1_consecutive_numbering_scheme' / 'main.tex'
print(text)
display_names = display_names_of_environments(text)
print(display_names)

\documentclass{article}
\usepackage{amsthm}

\newtheorem{theorem}{Theorem}
\newtheorem{lemma}[theorem]{Lemma}
\newtheorem{definition}[theorem]{Definition} % Note that `theorem`, `lemma`, and `definition` all have `theorem` as their counter.
\newtheorem{corollary}{Corollary} % Note that `corollary` has its own counter.
\newtheorem{remark}{Remark}[theorem] % `remark` has `theorem` as its counter
\newtheorem*{conjecture*}{Conjecture} % `conjecture*` has no counter

\begin{document}

\section{Introduction}

\begin{theorem}
This is Theorem 1.
\end{theorem}

\begin{lemma}
This is Lemma 2.
\end{lemma}

\begin{definition}
This is Definition 3.
\end{definition}

\end{document}
{'theorem': 'Theorem', 'lemma': 'Lemma', 'definition': 'Definition', 'corollary': 'Corollary', 'conjecture*': 'Conjecture', 'remark': 'Remark'}


In [7756]:
text = text_from_file(_test_directory() / 'latex_examples' / 'numbering_example_7_newtheorem_command_restarts_counter_by_section' / 'main.tex') 
print(text)
display_names = display_names_of_environments(text)
test_eq(display_names,
{'theorem': 'Theorem',
 'corollary': 'Corollary',
 'lemma': 'Lemma',
 'proposition': 'Proposition',})


% Based on an example from https://www.overleaf.com/learn/latex/Theorems_and_proofs#Numbered_theorems.2C_definitions.2C_corollaries_and_lemmas

\documentclass[12 pt]{amsart}

\newtheorem{theorem}{Theorem}[section]
\newtheorem{corollary}{Corollary}[theorem]
\newtheorem{lemma}[theorem]{Lemma}
% Note that the below invocation of \newtheorem is invalid:
% \newtheorem{proposition}[theorem]{Proposition}[section]
\newtheorem{proposition}{Proposition}[section]

\begin{document}
\section{Introduction}
Theorems can easily be defined:

\begin{theorem}
Let \(f\) be a function whose derivative exists in every point, then \(f\) is 
a continuous function.
\end{theorem}

\begin{theorem}[Pythagorean theorem]
\label{pythagorean}
This is a theorem about right triangles and can be summarised in the next 
equation 
\[ x^2 + y^2 = z^2 \]
\end{theorem}

And a consequence of theorem \ref{pythagorean} is the statement in the next 
corollary.

\begin{corollary}
There's no right rectangle whose sides measure 3c

In the following example, there are multiple `\newtheorem` commands defined in a single line. 

In [7757]:
text = r"""\theoremstyle{definition}                 \newtheorem{conj}{Conjecture}
\newtheorem*{example}{Example}            \newtheorem{defn}{Definition}
\newtheorem{remark}{Remark} \newtheorem*{notation}{Notation}
\begin{document}
\end{document}"""
test_eq(display_names_of_environments(text), {'conj': 'Conjecture', 'example': 'Example', 'defn': 'Definition', 'remark': 'Remark', 'notation': 'Notation'})

### Divide latex text into parts

In [7758]:
#| export
class DividedLatexPart(TypedDict):
    """
    Encapsulates a part divided from a latex document. Represents an entry within the list outputted by `divide_latex_text`.
    """
    note_title: str # often encapsulates the note type (i.e. section/subsection/display text of a theorem-like environment) along with the numbering. Sometimes `title` is just a number, which means that `text` is not of a `\section` or `\subsection` command and not of a theorem-like environment.
    text: str  # `text` is the text of the part

In [7759]:
#| export
def _setup_counters(
        numbertheorem_counters: dict[str, tuple[str, Union[str, None]]], # An output of `numbered_newtheorems_counters_in_preamble`
        ) -> dict[str, int]:
    r"""
    Return a dict whose keys are of counters in the LaTeX document and whose
    values are all `0`. These key-value pairs are used to keep track of
    the numberings of `parts`.

    One special key is the key of the empty string `''`, which counters the
    parts which do not get a numbering, i.e. for most text that lie outside
    of (numbered) environments

    """

    # cf. https://www.overleaf.com/learn/latex/Counters#Default_counters_in_LaTeX
    predefined_counters = [
        'part', # Incremented each time the `\part` command is used. It is not reset automatically and casn only be reset by the user
        'chapter', # Incremeneted each time the `\chapter` command is used.
        'section', # Incremented whenever a new `\section` command is encountered
        'subsection', # Incremented whenever a new `\subsection` command is encountered, reset whenever a new `\section` command is encountered
        'subsubsection', # Incremented whenever a new `\subsubsection` command is encounted, reset whenever a new `\subsection` or `\section` command is encountered
        'paragraph', # Incremeneted whenever a new paragraph is started. Reset whenever a new `\subsubsection`, `\subsection`, or `\section` command is encounted
        'subparagraph', # Incremented each time the `\subparagraph` command is used and reset at the beginning of a new
        'page', # Incremented each time a new page is started in the document
        'equation', # Incremeneted whenever the `\begin{equation}` environment is used. 
        'figure', # Incremented whenever a new `figure` environment is encountered
        'table', # Incremeneted whenever a new `taable` environment is encountered`
        'footnote', 
        'mpfootnote',
        'enumi',
        'enumii',
        'enumiii',
        'enumiv']

    counters = {counter: 0 for _, (counter, reset_counter) in numbertheorem_counters.items()}
    for counter in predefined_counters:
        counters[counter] = 0

    counters[''] = 0
    return counters

In [7760]:
#| hide
sample_counters = _setup_counters(
    {'thm': ('subsection', None), 'prop': ('subsection', None), 'cor': ('subsection', None), 'remark': ('remark', None), 'conj': ('subsection', None)})
assert 'remark' in sample_counters
test_eq(sample_counters['remark'], 0)
assert 'thm' not in sample_counters  # 'thm' is an environment name, but not a counter.

In [7761]:
#| export
# def _setup_numberwithins(
#         explicit_numberwithins: dict[str, str],
#         numbertheorem_counters: dict[str, tuple[str, Union[str, None]]], # An output of `numbered_newtheorems_counters_in_preamble`.
#         ) -> dict[str, str]: # The keys are counters and the values are all counters that the key is immediately numbered within.
#     """
#     Extracts information of counters that are reset when other counters are
#     incremented.

#     This is a helper function of `_setup_all_numberwithins` as well as
#     `divide_latex_text`.
#     """
#     builtin_numberwithins = {
#         'subsection': 'section',
#         'subsubsection': 'subsection',
#         'paragraph': 'subsubsection',
#         'subparagraph': 'paragraph',
#         'enumii': 'enumi',
#         'enumiii': 'enumii',
#         'enumiv': 'enumiii',
#         'part': 'chapter',
#         'appendix': 'chapter'
#     }
#     numberwithins = explicit_numberwithins | builtin_numberwithins

#     for environmentname, (counter, reset_by_counter) in numbertheorem_counters.items():
#         if reset_by_counter is None:
#             continue
#         numberwithins[environmentname] = reset_by_counter
#     return numberwithins

#| export
def _setup_numberwithins(
        explicit_numberwithins: dict[str, str],
        numbertheorem_counters: dict[str, tuple[str, Union[str, None]]],
        ) -> dict[str, str]:
    
    builtin_numberwithins = {
        'subsection': 'section',
        'subsubsection': 'subsection',
        'paragraph': 'subsubsection',
        'subparagraph': 'paragraph',
        'enumii': 'enumi',
        'enumiii': 'enumii',
        'enumiv': 'enumiii',
        'part': 'chapter',
        'appendix': 'chapter'
    }
    
    # 1. Start with built-ins, then merge explicit \numberwithin commands
    numberwithins = builtin_numberwithins | explicit_numberwithins

    # 2. Add counters from \newtheorem ONLY if they haven't been 
    # explicitly overridden by \numberwithin
    for env_name, (counter, reset_by_counter) in numbertheorem_counters.items():
        if reset_by_counter is not None:
            # ONLY update if the counter isn't already explicitly set 
            # (e.g., by explicit_numberwithins)
            if counter not in explicit_numberwithins:
                numberwithins[counter] = reset_by_counter
            
    return numberwithins 

In [7762]:
#| hide
from fastcore.test import test_eq

# 1. Test Default Hierarchy
# If no explicit commands are given, it should just return the built-ins
explicit = {}
nthm_counters = {}
withins = _setup_numberwithins(explicit, nthm_counters)

test_eq(withins['subsection'], 'section')
test_eq(withins['subsubsection'], 'subsection')

# 2. Test \newtheorem{thm}{Theorem}[section]
# This simulates the optional argument in \newtheorem
explicit = {}
nthm_counters = {'theorem': ('theorem', 'section')}
withins = _setup_numberwithins(explicit, nthm_counters)

test_eq(withins['theorem'], 'section')
test_eq(withins['subsection'], 'section') # Built-in still exists

# 3. Test \numberwithin Override (The 1.1.1 fix)
# Here, theorem is defined with [section] but overridden by \numberwithin{theorem}{subsection}
explicit = {'theorem': 'subsection'}
nthm_counters = {'theorem': ('theorem', 'section')}
withins = _setup_numberwithins(explicit, nthm_counters)

# CRITICAL: The explicit 'subsection' must win over the 'section' reset
test_eq(withins['theorem'], 'subsection')
test_eq(withins['subsection'], 'section')

# 4. Test priority merge (explicit | builtin)
# Verify that a custom environment using a standard name doesn't get wiped by built-ins
explicit = {'paragraph': 'section'} # Force paragraph to reset at section instead of subsubsection
nthm_counters = {}
withins = _setup_numberwithins(explicit, nthm_counters)

test_eq(withins['paragraph'], 'section')

In [7763]:
#| export
def _is_numberedwithin(
        counter_1, counter_2, numberwithins: dict[str, str]
        ) -> bool:
    """Return `True` if `counter_1` is numbered within `counter_2""" 
    if counter_1 not in numberwithins:
        return False
    elif numberwithins[counter_1] == counter_2:
        return True
    return _is_numberedwithin(
        numberwithins[counter_1], counter_2, numberwithins)

In [7764]:
#| export
# def _setup_all_numberwithins(
#         explicit_numberwithins: dict[str, str],
#         numbertheorem_counters: dict[str, tuple[str, Union[str, None]]], # An output of `numbered_newtheorems_counters_in_preamble`.
#         ) -> dict[str, list[str]]: # The keys are counters and the values are all counters that the key is numbered within.
#     """
#     This is a helper function of `divide_latex_text`.
#     """
#     numberwithins = _setup_numberwithins(explicit_numberwithins, numbertheorem_counters)
#     all_counters = set()
#     for key, value in numberwithins.items():
#         all_counters.add(key)
#         all_counters.add(value)
#     all_numbered_withins = {counter: [] for counter in all_counters}
#     for counter_1, counter_2 in product(all_counters, all_counters):
#         if _is_numberedwithin(counter_1, counter_2, numberwithins):
#             all_numbered_withins[counter_1].append(counter_2)
#     return all_numbered_withins

def _setup_all_numberwithins(explicit_numberwithins, numbertheorem_counters):
    """
    Ensures that parent_counter -> [child_counter1, child_counter2]
    is fully populated.

    This is a helper function of `divide_latex_text`.
    """
    all_withins = {}
    
    # 1. Process explicit resets (including your @addtoreset regex results)
    for child, parent in explicit_numberwithins.items():
        if parent not in all_withins:
            all_withins[parent] = []
        if child not in all_withins[parent]:
            all_withins[parent].append(child)
            
    # 2. Process resets from \newtheorem{thm}{Theorem}[section]
    for env, (counter, reset_by) in numbertheorem_counters.items():
        if reset_by:
            if reset_by not in all_withins:
                all_withins[reset_by] = []
            if counter not in all_withins[reset_by]:
                all_withins[reset_by].append(counter)
                
    return all_withins


In [7765]:
#| hide
# 1. Test basic @addtoreset / \numberwithin
sample_output = _setup_all_numberwithins({'equation': 'section'}, {})
# Section is the parent, equation is the child.
test_eq(sample_output['section'], ['equation']) 

# 2. Test theorem defined with [section]
sample_output = _setup_all_numberwithins({}, {'theorem': ('theorem', 'section')})
test_eq(sample_output['section'], ['theorem'])

# 3. Test combined resets
# Both equation and theorem are reset by section
sample_output = _setup_all_numberwithins({'equation': 'section'}, {'theorem': ('theorem', 'section')})
test_eq(sample_output['section'], ['equation', 'theorem'])

# 4. Test no parent
sample_output = _setup_all_numberwithins({}, {'theorem': ('theorem', None)})
# section should not even be a key here if nothing is reset by it
assert 'section' not in sample_output

In [7766]:
# #| hide
# sample_output = _setup_all_numberwithins({'equation': 'section'}, {})
# test_eq(sample_output['section'], [])
# test_eq(sample_output['subsection'], ['section'])
# test_eq(sample_output['equation'], ['section'])

# sample_output = _setup_all_numberwithins({'theorem': 'section'}, {})
# test_eq(sample_output['theorem'], ['section'])

# # In case that there is a `\newtheorem` invocation that also numbers the
# # theorem-like environment within some counter (e.g. `\newtheorem{theorem}{Theorem}[section]`),
# # we need to make sure that it is being setup like a numberwithin:
# sample_output = _setup_all_numberwithins({'equation': 'section'}, {'theorem': ('theorem', 'section')})
# test_eq(sample_output['theorem'], ['section'])
# test_eq(sample_output['equation'], ['section'])

# # In this example, let's say that we have a `\newtheorem{theorem}{Theorem}` instead
# sample_output = _setup_all_numberwithins({'equation': 'section'}, {'theorem': ('theorem', None)})
# assert 'theorem' not in sample_output
# test_eq(sample_output['equation'], ['section'])

In [7767]:
#| export
def _unnumbered_environments(
        numbertheorem_counters: dict[str, tuple[str, Union[str, None]]], # An output of `numbered_newtheorems_counters_in_preamble`
        display_names: dict[str, str]) -> set[str]:
    r"""Return the set of unnumbered theorem-like environments defined by
    `\newtheorem`.

    This is a helper function of `divide_latex_text`.
    """
    return {environment for environment in display_names
            if environment not in numbertheorem_counters}

    

In [7768]:
#| hide
sample_unnumbered_environments = _unnumbered_environments(
    {'theorem': ('theorem', None), 'lemma': ('theorem', None), 'definition': ('theorem', None), 'corollary': ('corollary', None), 'remark': ('theorem', None)},
    {'theorem': 'Theorem', 'lemma': 'Lemma', 'definition': 'Definition', 'corollary': 'Corollary', 'conjecture*': 'Conjecture', 'remark': 'Remark'} 
    )
test_eq(sample_unnumbered_environments, {'conjecture*'})

In [7769]:
#| export
# def _section_title(
#         text: str
#         ) -> tuple[bool, str]: # The bool is `True` if the section/subsection is numbered (i.e. is `section` or `subsection` as opposed to `section*` or `subsection*`). The `str` is the title of the section or subsection
#     """Return the title of a section or subsection from a latex str
#     and whether or not the section/subsection is numbered"""

#     # Note that the `section` command has the optional argument `toc-title` which appears
#     # in the table of contents, cf.
#     # http://latexref.xyz/_005csection.html
#     # pattern = regex.compile(
#     #     r'\\(?:section|subsection|subsubsection)\s*(?:\[.*\])?(\*)?\s*'
#     #     r'\{((?>[^{}]+|\{(?2)\})*)\}',
#     #     regex.MULTILINE
#     # )
#     regex_search = regex.search(SECTION_LIKE_PATTERN, text)
#     is_numbered = regex_search.group(1) is None
#     title = regex_search.group(2)
#     return is_numbered, title

#| export
def _section_title(
        text: str
        ) -> tuple[bool, str]: # The bool is `True` if the section/subsection is numbered (i.e. is `section` or `subsection` as opposed to `section*` or `subsection*`). The `str` is the title of the section or subsection
    """Return the title of a section or subsection from a latex str
    and whether or not the section/subsection is numbered"""

    # Note that the `section` command has the optional argument `toc-title` which appears
    # in the table of contents, cf.
    # http://latexref.xyz/_005csection.html
    
    # Use lstrip() to ensure leading whitespace/newlines don't prevent a match
    regex_search = regex.search(SECTION_LIKE_PATTERN, text.lstrip())
    
    # Handle the case where the regex fails to match the input string
    if not regex_search:
        return False, text
        
    is_numbered = regex_search.group(1) is None
    title = regex_search.group(2)
    return is_numbered, title

In [7770]:
#| hide
def test_regex_match():
    text = r"\part{Algebra}"
    print(f"Pattern: {SECTION_LIKE_PATTERN.pattern}")
    match = regex.search(SECTION_LIKE_PATTERN, text)
    print(f"Regex Match Found: {match is not None}")
    if match:
        print(f"Group 1 (Star): {match.group(1)}")
        print(f"Group 2 (Title): {match.group(2)}")
    
    is_num, title = _section_title(text)
    print(f"Section Title Result: is_numbered={is_num}, title='{title}'")
    
    assert is_num == True
    assert title == "Algebra"

test_regex_match()

Pattern: \\(?:part|section|subsection|subsubsection)\s*(?:\[.*\])?(\*)?\s*\{((?>[^{}]+|\{(?2)\})*)\}
Regex Match Found: True
Group 1 (Star): None
Group 2 (Title): Algebra
Section Title Result: is_numbered=True, title='Algebra'


In [7771]:
#| hide

# subsection, no extraneous spaces
sample_section = _section_title(r"\subsection{I am a subsection}")
test_eq(sample_section, (True, 'I am a subsection'))

# section, with extraneous spaces
sample_section = _section_title(r"\section {Generating series of special divisors}")
test_eq(sample_section, (True, 'Generating series of special divisors'))

# section, unnumbered
sample_section = _section_title(r"\section*{I am an unnumbered section}")
test_eq(sample_section, (False, 'I am an unnumbered section'))

# Subsection, unnumbered, extraneous spaces
sample_section = _section_title(r"\subsection*    {I am an unnumbered section and I have extraneous spaces}")
test_eq(sample_section, (False, 'I am an unnumbered section and I have extraneous spaces'))

# Multiline section
sample_section = _section_title(
    r"""\section*    {I am a section and I have span 
    multiple lines}""")
test_eq(sample_section, (False, 'I am a section and I have span \n    multiple lines'))

# Section with curly braces
sample_section = _section_title(
    r"""\section{ Can I talk about the finite field \mathcal{F}_p in this title?
        Can I also have multiple lines? Yes I can!}"""
)
test_eq(sample_section, (True, r""" Can I talk about the finite field \mathcal{F}_p in this title?
        Can I also have multiple lines? Yes I can!"""))

# Section with table of contents
sample_section = _section_title(
    r"\section [This is a Table of contents title] {This is the section title}"
)
test_eq(sample_section, (True, r"""This is the section title"""))

# # Section, also multiline
# sample_section = _section_title(
#     r"""\section{Exceptional maximal subgroups of 
# \texorpdfstring{\(\GSp_4(\ff_\ell)\)}{GSp4Fell}}"""
# )
# sample_section[1]

In [7772]:
#| export
def _is_section_node(node: LatexNode):
    return (node.isNodeType(LatexMacroNode)
            and node.macroname == 'section')

def _is_subsection_node(node: LatexNode):
    return (node.isNodeType(LatexMacroNode)
            and node.macroname == 'subsection')

def _is_subsubsection_node(node: LatexNode):
    return (node.isNodeType(LatexMacroNode)
            and node.macroname == 'subsubsection')

def _is_environment_node(node: LatexNode):
    return node.isNodeType(LatexEnvironmentNode)

def _text_is_of_section_like_node(text: str):
    """Return `True` if `text` represents the text for a section node.

    In principal, this function should act like 
    `_is_section_node or _is_subsection_node or _is_subsubsection_node`
    except that it takes a `str` as its argument instead of a `LatexNode`.
    This function is
    implemented using a regex pattern instead of using
    `_is_section_node` to save time.
    """
    return bool(regex.match(SECTION_LIKE_PATTERN, text.lstrip()))


def _text_is_of_environment_node(text: str):
    """Return `True` if `text` represents an environment node
    (at least at the start).
    
    In principal, this function should act like `_is_environment_node`
    except that it takes a `str` as its argument instead of a `LatexNode`.
    This function is implemented using a regex pattern instead of using
    '_is_environment_node` to save time.
    """
    return bool(regex.match(ENVIRONMENT_PATTERN, text.lstrip()))


def _environment_name_of_text(text: str):
    """Return `True` if `text` represents an environment node
    (at least at the start).
    
    Assumes that `_text_is_of_environment_node(text)` is `True`.
    """
    match = regex.match(ENVIRONMENT_PATTERN, text.lstrip())
    return match.group(1)

In [7773]:
#| hide
text = r"""
\documentclass{article}

\theoremstyle{plain}
\newtheorem{theorem}{Theorem}

\begin{document}

\section{This is section 1}

\subsection{This is subsection 1.1}

\begin{theorem}
\end{theorem}

\end{document}
"""
document_node = find_document_node(text)
assert _is_section_node(document_node.nodelist[1])
assert not _is_section_node(document_node.nodelist[3])
assert not _is_section_node(document_node.nodelist[5])

assert not _is_subsection_node(document_node.nodelist[1])
assert _is_subsection_node(document_node.nodelist[3])
assert not _is_subsection_node(document_node.nodelist[5])

assert not _is_environment_node(document_node.nodelist[1])
assert not _is_environment_node(document_node.nodelist[3])
assert _is_environment_node(document_node.nodelist[5])

# for node in document_node.nodelist:
#     print('\n')
#     print(node)
#     if node.isNodeType(LatexMacroNode):
#         print(node.macroname)
#     elif node.isNodeType(LatexEnvironmentNode):
#         print(node.environmentname)

In [7774]:
#| hide
assert _text_is_of_section_like_node(r"\section {Generating series of special divisors}")
assert _text_is_of_section_like_node(r"   \section {Generating series of special divisors}")
assert _text_is_of_section_like_node(r"\subsection{I am a subsection}")
assert _text_is_of_section_like_node(r"\section*{I am an unnumbered section}")
assert _text_is_of_section_like_node(r"\subsection*    {I am an unnumbered section and I have extraneous spaces}")
assert _text_is_of_section_like_node(r"""\section*    {I am a section and I have span 
    multiple lines}""")
assert _text_is_of_section_like_node(r"""\section{ Can I talk about the finite field \mathcal{F}_p in this title?
        Can I also have multiple lines? Yes I can!}""")
assert _text_is_of_section_like_node(r"""\subsubsection{Hi}""")

assert not _text_is_of_section_like_node(r"""Something something""")
assert not _text_is_of_section_like_node(r"""\begin{theorem}""")
assert not _text_is_of_section_like_node(r"hi \section{title}")

assert _text_is_of_environment_node(r"""\begin{theorem} blah blah blah""")
assert _text_is_of_environment_node(r"""\begin{theorem} blah blah blah \end{theorem}""")
assert not _text_is_of_environment_node(r"""hi""")
_environment_name_of_text(r"""\begin{theorem} blah blah blah""")

'theorem'

In [7775]:
#| export
def _is_part_node(node: LatexNode):
    return (node.isNodeType(LatexMacroNode) and node.macroname == 'part')

In [7776]:
#| export
# def _is_numbered(
#         node: LatexNode,
#         numbertheorem_counters: dict[str, str]
#         ) -> bool:
#     # Ensure _is_part_node is part of the check
#     if (_is_part_node(node) or _is_section_node(node) or 
#         _is_subsection_node(node) or _is_subsubsection_node(node)):
#         is_numbered, _ = _section_title(node.latex_verbatim())
#         return is_numbered
#     elif _is_environment_node(node):
#         return node.environmentname in numbertheorem_counters
#     else:
#         return False

def _is_numbered(
        node: LatexNode,
        numbertheorem_counters: dict[str, tuple[str, Union[str, None]]]
        ) -> bool:
    # 1. Structural checks
    structural_macros = ['part', 'section', 'subsection', 'subsubsection']
    
    if isinstance(node, LatexMacroNode) and node.macroname in structural_macros:
        # We need to look at the source text context. 
        # LatexMacroNode has 'pos' (start) and 'len' (length of macro).
        # We check if the character at 'pos + len' in the original LaTeX string is '*'
        parsing_state = getattr(node, 'parsing_state', None)
        if parsing_state and hasattr(parsing_state, 's'):
            full_text = parsing_state.s
            after_macro_pos = node.pos + node.len
            if after_macro_pos < len(full_text) and full_text[after_macro_pos] == '*':
                return False
        
        # Fallback: check the verbatim text of the node itself
        verbatim = node.latex_verbatim()
        if '*' in verbatim[len(node.macroname):]:
            return False
            
        return True
            
    # 2. Environment checks
    if isinstance(node, LatexEnvironmentNode):
        return node.environmentname in numbertheorem_counters
        
    return False

In [7777]:
#| hide
text = r"""
\documentclass{article}

\theoremstyle{plain}
\newtheorem{theorem}{Theorem}
\newtheorem*{theorem*}{Theorem}

\begin{document}
\begin{theorem}
\end{theorem}
\begin{theorem*}
\end{theorem*}
\end{document}
"""
document_node = find_document_node(text)
environments_to_counters = {'theorem': 'theorem'}

assert _is_numbered(document_node.nodelist[1], environments_to_counters)
assert not _is_numbered(document_node.nodelist[2], environments_to_counters)


# Example with numberwithin specified.
text = r"""
\documentclass{article}

\numberwithin{equation}{section}

\theoremstyle{plain}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem*{theorem*}{Theorem}

\begin{document}
\begin{theorem}
\end{theorem}
\begin{theorem*}
\end{theorem*}
\end{document}
"""

document_node = find_document_node(text)
environments_to_counters = {'theorem': 'section'}

assert _is_numbered(document_node.nodelist[1], environments_to_counters)
assert not _is_numbered(document_node.nodelist[2], environments_to_counters)

# Example for sections and subsections
text = r"""
\begin{document}
\section{Section 1}
\subsection*{Unnumbered section}
\end{document}
"""
document_node = find_document_node(text)
environments_to_counters = {}

assert _is_numbered(document_node.nodelist[1], environments_to_counters)
assert not _is_numbered(document_node.nodelist[2], environments_to_counters)

In [7778]:
#| export
def get_node_from_simple_text(
        text: str) -> LatexNode:
    """Return the (first) `LatexNode` object from a str."""
    w = LatexWalker(text)
    nodelist, _, _ = w.get_latex_nodes(pos=0)
    return nodelist[0]



In [7779]:
#| hide
def test_part_recognition():
    # Standard part
    node_part = get_node_from_simple_text(r"\part{Algebra}")
    is_num = _is_numbered(node_part, {})
    print(f"\\part numbered: {is_num}")
    assert is_num == True

    # Starred part
    node_part_star = get_node_from_simple_text(r"\part*{Appendix}")
    print(node_part_star)
    is_num_star = _is_numbered(node_part_star, {})
    print(f"\\part* numbered: {is_num_star}")
    assert is_num_star == False

test_part_recognition()

\part numbered: True
LatexMacroNode(parsing_state=<parsing state 2284992509376>, pos=0, len=5, macroname='part', nodeargd=ParsedMacroArgs(argspec='', argnlist=[]), macro_post_space='')
\part* numbered: False


In [7780]:
#| hide
def test_node_type():
    node = get_node_from_simple_text(r"\part{Algebra}")
    print(f"Node type: {type(node)}")
    if isinstance(node, LatexMacroNode):
        print(f"Macro name: {node.macroname}")
    print(f"Is part node helper: {_is_part_node(node)}")
    
    assert _is_part_node(node) == True

test_node_type()

Node type: <class 'pylatexenc.latexwalker.LatexMacroNode'>
Macro name: part
Is part node helper: True


In [7781]:
#| hide
def test_increment_logic():
    node = get_node_from_simple_text(r"\part{Algebra}")
    is_part = _is_part_node(node)
    is_num = _is_numbered(node, {})
    
    print(f"Is Part Node: {is_part}")
    print(f"Is Numbered: {is_num}")
    
    # In _change_counters, the logic is usually:
    # if is_part: counter = 'part'
    # if is_num and counter: counters[counter] += 1
    
    assert is_part and is_num

test_increment_logic()

Is Part Node: True
Is Numbered: True


In [7782]:
#| export
def text_from_node(
        node: LatexNode) -> str:
    """Return the str representing `node`."""
    return node.latex_verbatim()
    # l2t = LatexNodes2Text()
    # if node.isNodeType('text'):
    #     return node.chars
    # else:
    #     full_text = ''
    #     for child_node in node.children:
    #         full_text += text_from_node(child_node)
    #     return full_text
    # return LatexNodes2Text().node_to_text(node)

In [7783]:
#| hide
def test_counter_evolution():
    numbertheorem_counters = {'remark': ('equation', 'section')}
    all_numberwithins = {'part': ['section'], 'section': ['equation']}
    counters = _setup_counters(numbertheorem_counters)
    
    # Step 1: Process Part
    node_part = get_node_from_simple_text(r"\part{Algebra}")
    _change_counters(node_part, counters, numbertheorem_counters, all_numberwithins)
    print(f"After Part: {counters['part']=}") # Should be 1
    
    # Step 2: Process Section
    node_sec = get_node_from_simple_text(r"\section{Modules}")
    _change_counters(node_sec, counters, numbertheorem_counters, all_numberwithins)
    print(f"After Section: {counters['section']=}") # Should be 1
    
    # Step 3: Process Remark
    node_rem = get_node_from_simple_text(r"\begin{remark}...\end{remark}")
    _change_counters(node_rem, counters, numbertheorem_counters, all_numberwithins)
    print(f"After Remark: {counters['equation']=}") # Should be 1
    
    assert counters['part'] == 1
    assert counters['section'] == 1
    assert counters['equation'] == 1

test_counter_evolution()

After Part: counters['part']=1
After Section: counters['section']=1
After Remark: counters['equation']=1


In [7784]:
node = get_node_from_simple_text(
    r"""\begin{theorem}
lalalala
\begin{equation}
\end{equation}
\end{theorem}"""
)
text_from_node(node)

'\\begin{theorem}\nlalalala\n\\begin{equation}\n\\end{equation}\n\\end{theorem}'

In [7785]:
text = r"""\begin{thm}This is a theorem. \end{thm}"""
node = get_node_from_simple_text(text)
assert isinstance(node, LatexEnvironmentNode)
test_eq(node.environmentname, 'thm')


text = r"""\begin{thm}This is a theorem. \end{thm} \begin{proof} This is a proof. It is not captured by the `get_node_from_simple_text` function \end{proof}"""
node = get_node_from_simple_text(text)
assert isinstance(node, LatexEnvironmentNode)
test_eq(node.environmentname, 'thm')

In [7786]:
#| export
def _recursive_reset(counter_name, counters, all_numberwithins):
    """Resets all descendant counters in the hierarchy."""
    if counter_name in all_numberwithins:
        for child in all_numberwithins[counter_name]:
            # Reset the child
            counters[child] = 0
            # Continue down the tree (e.g., section -> subsection -> subsubsection)
            _recursive_reset(child, counters, all_numberwithins)

In [7787]:
def test_recursive_reset():
    # Setup: 
    # section -> [subsection, equation]
    # subsection -> [subsubsection]
    all_withins = {
        'section': ['subsection', 'equation'],
        'subsection': ['subsubsection']
    }

    # Test Case 1: Shallow reset (resetting a leaf node)
    # Equation has no children, so only equation should change.
    counters = {'section': 1, 'subsection': 2, 'equation': 5}
    _recursive_reset('equation', counters, all_withins)
    test_eq(counters, {'section': 1, 'subsection': 2, 'equation': 5}) 
    # Note: _recursive_reset resets CHILDREN, not the counter itself. 
    # The increment of the counter happens in _change_counters.

    # Test Case 2: Multi-level reset (The "Waterfall")
    # Resetting 'section' should zero out EVERYTHING below it.
    counters = {
        'section': 1, 
        'subsection': 2, 
        'subsubsection': 3, 
        'equation': 10
    }
    _recursive_reset('section', counters, all_withins)
    
    test_eq(counters['section'], 1)       # Parent stays same
    test_eq(counters['subsection'], 0)    # Direct child reset
    test_eq(counters['equation'], 0)      # Direct child reset
    test_eq(counters['subsubsection'], 0) # Grandchild reset (via subsection)

    # Test Case 3: Intermediate reset
    # Resetting 'subsection' should reset 'subsubsection' but NOT 'section' or 'equation'
    counters = {
        'section': 1, 
        'subsection': 2, 
        'subsubsection': 3, 
        'equation': 10
    }
    _recursive_reset('subsection', counters, all_withins)
    
    test_eq(counters['section'], 1)
    test_eq(counters['equation'], 10)
    test_eq(counters['subsubsection'], 0)

    # Test Case 4: Missing counters
    # If a counter isn't in the dict yet, the function should handle it gracefully
    # (assuming counters.get() or just initializing to 0)
    counters = {'section': 1}
    # Should not raise KeyError even if 'subsection' is missing from 'counters'
    # but present in 'all_withins'
    _recursive_reset('section', counters, all_withins)
    test_eq(counters['subsection'], 0)
test_recursive_reset()

In [7788]:
#| hide
def test_recursive_reset_logic():
    # Initial state: Section 1, Subsection 2, Equation 5
    counters = {'section': 1, 'subsection': 2, 'equation': 5}
    
    # CORRECT MAPPING: Parent -> [Children]
    # Section is the parent of subsection and equation
    all_withins = {
        'section': ['subsection', 'equation']
    }
    
    # Simulate encountering a new \section
    node_sec = get_node_from_simple_text(r"\section{New}")
    nthm_counters = {'thm': ('equation', None)}
    
    _change_counters(node_sec, counters, nthm_counters, all_withins)
    
    # Verification
    test_eq(counters['section'], 2)    # Incremented
    test_eq(counters['subsection'], 0) # Reset by section
    test_eq(counters['equation'], 0)   # Reset by section

test_recursive_reset_logic()

In [7789]:
#| hide
#| hide
from fastcore.test import test_eq

# Setup a dirty state
dirty_counts = {'section': 1, 'subsection': 4, 'theorem': 12, 'equation': 9}

# CORRECT MAPPING: Parent -> [List of Children]
all_withins = {
    'section': ['subsection'],           # section resets subsection
    'subsection': ['theorem', 'equation'] # subsection resets theorem and equation
}

# Now, resetting 'section' will find 'subsection', 
# which in turn will recursively find 'theorem' and 'equation'.
_recursive_reset('section', dirty_counts, all_withins)

test_eq(dirty_counts['subsection'], 0)
test_eq(dirty_counts['theorem'], 0)
test_eq(dirty_counts['equation'], 0)
test_eq(dirty_counts['section'], 1)
# from fastcore.test import test_eq

# # Setup a dirty state where counts exist in a deep hierarchy
# dirty_counts = {'section': 1, 'subsection': 4, 'theorem': 12, 'equation': 9}
# # all_numberwithins uses lists as per your original implementation
# all_withins = {
#     'subsection': ['section'], 
#     'theorem': ['subsection'], 
#     'equation': ['subsection']
# }

# # Resetting the section should cascade down to subsections AND their children (theorems/equations)
# _recursive_reset('section', dirty_counts, all_withins)

# test_eq(dirty_counts['subsection'], 0)
# test_eq(dirty_counts['theorem'], 0)
# test_eq(dirty_counts['equation'], 0)
# # The parent itself should NOT be reset by this function (it was incremented elsewhere)
# test_eq(dirty_counts['section'], 1)

In [7790]:
#| export
def _change_counters(
        node,
        counters,
        numbertheorem_counters: dict[str, tuple[str, str]],
        all_numberwithins: dict[str, list[str]]
        ):
    
    """Preliminarily update the counters for `node`, but not for
    any of its subnodes. This is mostly for
    theoremlike environments and for sectionlike environments.
    
    Helper function to `_process_node`.
    """
    counter = None
    # Identify the counter based on node type/macro name
    if _is_environment_node(node):
        env_name = node.environmentname
        if env_name in numbertheorem_counters:
            counter, _ = numbertheorem_counters[env_name]
            
    elif _is_part_node(node): # ADD THIS
        counter = 'part'
    elif _is_section_node(node):
        counter = 'section'
    elif _is_subsection_node(node):
        counter = 'subsection'
    elif _is_subsubsection_node(node):
        counter = 'subsubsection'

    # Check if the node is numbered (e.g., handles \section vs \section*)
    is_numbered = _is_numbered(node, numbertheorem_counters)

    # Inside _change_counters
    if is_numbered and counter:
        counters[counter] = counters.get(counter, 0) + 1
        # Check if 'counter' here is 'section'
        _recursive_reset(counter, counters, all_numberwithins)

    # if is_numbered and counter:
    #     counters[counter] = counters.get(counter, 0) + 1
    #     # Trigger the recursive reset for children (e.g., section resets equation)
    #     _recursive_reset(counter, counters, all_numberwithins)


In [7791]:
#| hide
def test_change_counters_logic():
    # Setup initial state
    numbertheorem_counters = {
        'theorem': ('equation', None), 
        'cor': ('equation', None),
        'lemma': ('lemma', 'section')
    }
    all_withins = {
        'section': ['subsection', 'equation', 'lemma'], # Parent: [Children]
        'subsection': ['subsubsection']
    }
    
    # Test 1: Section increment and recursive reset
    counters = {'section': 1, 'subsection': 4, 'equation': 12, 'lemma': 5}
    node_sec = get_node_from_simple_text(r"\section{Introduction}")
    _change_counters(node_sec, counters, numbertheorem_counters, all_withins)
    
    test_eq(counters['section'], 2)
    test_eq(counters['subsection'], 0)
    test_eq(counters['equation'], 0)
    test_eq(counters['lemma'], 0)

    # Test 2: Theorem sharing 'equation' counter
    # (Continuing from previous state where equation is 0)
    node_thm = get_node_from_simple_text(r"\begin{theorem} content \end{theorem}")
    _change_counters(node_thm, counters, numbertheorem_counters, all_withins)
    test_eq(counters['equation'], 1)
    
    # Test 3: Shared counter incremented by different environment
    node_cor = get_node_from_simple_text(r"\begin{cor} content \end{cor}")
    _change_counters(node_cor, counters, numbertheorem_counters, all_withins)
    test_eq(counters['equation'], 2) # Incremented from 1 to 2

    # Test 4: Subsubsection identification and increment
    counters['subsubsection'] = 1
    node_subsub = get_node_from_simple_text(r"\subsubsection{Title}")
    _change_counters(node_subsub, counters, numbertheorem_counters, all_withins)
    test_eq(counters['subsubsection'], 2)

    # Test 5: Unnumbered Star-variants (should NOT increment)
    pre_star_counters = counters.copy()
    node_sec_star = get_node_from_simple_text(r"\section*{Introduction}")
    _change_counters(node_sec_star, counters, numbertheorem_counters, all_withins)
    
    node_thm_star = get_node_from_simple_text(r"\begin{theorem*} content \end{theorem*}")
    _change_counters(node_thm_star, counters, numbertheorem_counters, all_withins)
    
    test_eq(counters, pre_star_counters)

    # Test 6: Deep Reset (Section -> Subsection -> Subsubsection)
    counters = {'section': 1, 'subsection': 1, 'subsubsection': 1}
    node_sec_new = get_node_from_simple_text(r"\section{Next}")
    _change_counters(node_sec_new, counters, numbertheorem_counters, all_withins)
    
    test_eq(counters['section'], 2)
    test_eq(counters['subsection'], 0)
    test_eq(counters['subsubsection'], 0) # Triggered via subsection reset
test_change_counters_logic()

In [7792]:
#| export
def _subsubnodes(
        subnode,
        ) -> list[LatexNode]: 
    """Find subnodes of `subnode` to add to the queue

    Helper function to `_change_counters_antecedently`.
    """ 
    if not hasattr(subnode, 'nodelist'):
        return []
    return subnode.nodelist

In [7793]:
#| export
def _update_counter_for_subsubnodes(
        subnode,
        counters,
        numbertheorem_counters: dict[str, str],
        all_numberwithins: dict[str, list[str]],
        ) -> None: 
    """Iterate through the immediate subnodes of `subnode` to see if the
    counter needs to be updated for any
    
    Helper function to `_change_counters_antecedently`.
    """ 
    if not hasattr(subnode, 'nodelist'):
        return
    for subsubnode in subnode.nodelist:
       _change_counters(subsubnode, counters, numbertheorem_counters, all_numberwithins)
            

In [7794]:
#| export
def _change_counters_antecedently(
        node,
        counters,
        numbertheorem_counters: dict[str, str],
        all_numberwithins: dict[str, list[str]],
        ):
    """Update the counters to account for any environments contained within
    `node`, but not `node` itself.

    This is mostly for theorem-like environments which share a counter with 
    something like the `equation` environment; sometimes a theorem-like
    environment can have an `equation` environment within it. Note that
    the `_process_node` function alreay invokes `_change_counters` on
    `node`, so the counter is already updated for `node` by the time
    `_change_counters_antecedently` is invoked.

    Helper function to `_process_node`.
    """
    queue = [node]
    while queue:
        subnode = queue.pop()
        _update_counter_for_subsubnodes(
            subnode, counters, numbertheorem_counters, all_numberwithins)
        queue.extend(_subsubnodes(subnode))



In [7795]:
#| hide

# text = r"""\begin{thm}This is a theorem. \end{thm}"""
# node = get_node_from_simple_text(text)
# # Test a theoreem being counted by its own counter.
# numbertheorem_counters = {'thm': ('thm', None)}
# all_numberwithins = {}
# counters = {'thm': 1}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'thm': 2})
# # Test a theorem being countered by the equation counter.
# numbertheorem_counters = {'thm': ('equation', None)}
# all_numberwithins = {}
# counters = {'equation': 2}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'equation': 3})

# text = r"""\begin{corollary}This is a corollary. \end{orollary}"""
# node = get_node_from_simple_text(text)
# # Test a theorem-like environment being counted by the counter of
# # another theorem-like environment
# numbertheorem_counters = {'corollary': ('theorem', None), 'theorem': ('theorem', None)}
# all_numberwithins = {}
# counters = {'theorem': 0}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'theorem': 1})


# # Test a theorem-like environment whose counter is numbered within
# # The section counter.
# # First, see what happens when a theorem is called
# text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('theorem', None)}
# all_numberwithins = {'section': ['theorem']}
# # all_numberwithins = {'theorem': ['section']}
# counters = {'section': 1, 'theorem': 0}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'theorem': 1})
# # Next, see what happens when a new section is invoked:
# text = r"""\section{New section! The theorem counter should be reset}"""
# node = get_node_from_simple_text(text)
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 2, 'theorem': 0})

# # Test a theorem-like environment sharing a counter with equation
# # and in turn equation is numbered within section.
# text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('equation', None)}
# all_numberwithins = {'equation': ['section']}
# counters = {'section': 1, 'equation': 0}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'equation': 1})
# # Next, see what happens when a new section is invoked:
# text = r"""\section{New section! The theorem counter should be reset}"""
# node = get_node_from_simple_text(text)
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 2, 'equation': 0})

# # Test an unnumbered theorem-like environment counter
# text = r"""\begin{thm*}This is a theorem. \end{thm*}"""
# node = get_node_from_simple_text(text)
# # Test a theoreem being counted by its own counter.
# numbertheorem_counters = {'thm': ('thm', None)}
# all_numberwithins = {}
# counters = {'thm': 1}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'thm': 1})

# # Test a theorem-like environment sharing a counter with equation
# # and in turn equation is numbered within section, but the 
# # environment is unnumbered.
# text = r"""\begin{theorem*}This is a theorem. \end{theorem*}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('equation', None)}
# all_numberwithins = {'equation': ['section']}
# counters = {'section': 1, 'equation': 0}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'equation': 0})
# # Next, see what happens when a unnumbered new section is invoked:
# text = r"""\section*{New section! The theorem counter should be reset}"""
# node = get_node_from_simple_text(text)
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'equation': 0})

# # Test the counter for text that does not belong to an environment
# # In the current implementation of _change_counters, the '' counter
# # is not actually changed.
# text = r"""Just some text."""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('equation', None)}
# all_numberwithins = {'equation': ['section']}
# counters = {'section': 1, 'equation': 0, '': 0}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'equation': 0, '': 0})

# # Test the counter for theorems the share a counter with subsubsection
# text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('subsubsection', None)}
# all_numberwithins = {}
# counters = {'section': 1, 'subsubsection': 1}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'subsubsection': 2})
# # Next, see what happens when a new subsection is invoked:
# text = r"""\subsubsection{New subsubsection! The theorem counter should be reset}"""
# node = get_node_from_simple_text(text)
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'subsubsection': 3})

In [7796]:
#| hide

# --- Standard Incrementation Tests ---

text = r"""\begin{thm}This is a theorem. \end{thm}"""
node = get_node_from_simple_text(text)
# Test a theorem being counted by its own counter.
numbertheorem_counters = {'thm': ('thm', None)}
all_numberwithins = {}
counters = {'thm': 1}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'thm': 2})

# Test a theorem being countered by the equation counter.
numbertheorem_counters = {'thm': ('equation', None)}
all_numberwithins = {}
counters = {'equation': 2}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'equation': 3})

text = r"""\begin{corollary}This is a corollary. \end{orollary}"""
node = get_node_from_simple_text(text)
# Test a theorem-like environment being counted by the counter of another environment
numbertheorem_counters = {'corollary': ('theorem', None), 'theorem': ('theorem', None)}
all_numberwithins = {}
counters = {'theorem': 0}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'theorem': 1})


# --- Hierarchy and Reset Tests ---

# Test a theorem-like environment whose counter is numbered within the section counter.
text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'theorem': ('theorem', None)}
# Correct: parent 'section' resets child 'theorem'
all_numberwithins = {'section': ['theorem']} 
counters = {'section': 1, 'theorem': 0}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'theorem': 1})

# Next, see what happens when a new section is invoked:
text = r"""\section{New section!}"""
node = get_node_from_simple_text(text)
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 2, 'theorem': 0})


# Test a theorem sharing a counter with equation, reset by section.
text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'theorem': ('equation', None)}
# Correct: parent 'section' resets child 'equation'
all_numberwithins = {'section': ['equation']}
counters = {'section': 1, 'equation': 0}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'equation': 1})

# Next, see what happens when a new section is invoked:
text = r"""\section{New section!}"""
node = get_node_from_simple_text(text)
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 2, 'equation': 0})


# --- Unnumbered Variants Tests ---

# Test an unnumbered theorem-like environment counter (thm*)
text = r"""\begin{thm*}This is a theorem. \end{thm*}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'thm': ('thm', None)}
all_numberwithins = {}
counters = {'thm': 1}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'thm': 1})

# Test unnumbered environment with hierarchy
text = r"""\begin{theorem*}This is a theorem. \end{theorem*}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'theorem': ('equation', None)}
all_numberwithins = {'section': ['equation']}
counters = {'section': 1, 'equation': 0}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'equation': 0})

# Test unnumbered section (section*)
text = r"""\section*{New section!}"""
node = get_node_from_simple_text(text)
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'equation': 0})


# --- Edge Cases and Sub-sections ---

# Test text that does not belong to an environment
text = r"""Just some text."""
node = get_node_from_simple_text(text)
counters = {'section': 1, 'equation': 0, '': 0}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'equation': 0, '': 0})

# Test the counter for theorems that share a counter with subsubsection
text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'theorem': ('subsubsection', None)}
# Correct hierarchy: section resets subsection, subsection resets subsubsection
all_numberwithins = {'section': ['subsection'], 'subsection': ['subsubsection']}
counters = {'section': 1, 'subsection': 1, 'subsubsection': 1}
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'subsection': 1, 'subsubsection': 2})

# Next, see what happens when a new subsubsection is invoked:
text = r"""\subsubsection{New subsubsection!}"""
node = get_node_from_simple_text(text)
_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'subsection': 1, 'subsubsection': 3})

In [7797]:
#| hide

# # Here, we have that equations are numbered within sections and theorems share the equation's counter. 
# # We also have a theorem environment that houses an equation environment.
# text = r"""\begin{theorem}
# This is theorem 1.1
# \begin{equation}
# asdf
# \end{equation}
# \end{theorem}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('equation', None), 'equation': ('equation', None)}
# all_numberwithins = {'equation': ['section'], 'figure': ['section']}
# counters = {'section': 1, 'equation': 0}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'equation': 1})
# # Next, see what happens when `_change_counters_antecedently` is invoked to inspect the
# # subnodes.
# _change_counters_antecedently(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'equation': 2})

# # Everything in the below example is the same as in the previous example,
# # except the theorem environment houses an enumerate environment, which in turn houses
# # an equation environment.
# text = r"""\begin{theorem}
# This is theorem 1.1
# \begin{enumerate}
# \item \begin{equation}
# asdf
# \end{equation}
# \end{enumerate}
# \end{theorem}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('equation', None), 'equation': ('equation', None)}
# all_numberwithins = {'equation': ['section'], 'figure': ['section']}
# counters = {'section': 1, 'equation': 0}
# _change_counters(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'equation': 1})
# # Next, see what happens when `_change_counters_antecedently` is invoked to inspect the
# # subnodes.
# _change_counters_antecedently(node, counters, numbertheorem_counters, all_numberwithins)
# test_eq(counters, {'section': 1, 'equation': 2})

In [7798]:
# --- Corrected Test Block ---
# --- Example 1: Direct Nesting ---
text = r"""\begin{theorem}
This is theorem 1.1
\begin{equation}
asdf
\end{equation}
\end{theorem}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'theorem': ('equation', None), 'equation': ('equation', None)}

# FIXED: section resets equation, not the other way around
all_numberwithins = {'section': ['equation', 'figure']}
counters = {'section': 1, 'equation': 0}

_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'equation': 1})

# Increments for the nested equation
_change_counters_antecedently(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'equation': 2})


# --- Example 2: Deep Nesting (via enumerate) ---
text = r"""\begin{theorem}
This is theorem 1.1
\begin{enumerate}
\item \begin{equation}
asdf
\end{equation}
\end{enumerate}
\end{theorem}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'theorem': ('equation', None), 'equation': ('equation', None)}
all_numberwithins = {'section': ['equation', 'figure']}
counters = {'section': 1, 'equation': 0}

_change_counters(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'equation': 1})

# This verifies traversal through intermediate nodes like 'enumerate' and 'item'
_change_counters_antecedently(node, counters, numbertheorem_counters, all_numberwithins)
test_eq(counters, {'section': 1, 'equation': 2})

In [7799]:
#| export
def _numbering_helper(
        trailing_numbering: str,
        counter: str,
        numberwithins: dict[str, str],
        counters: dict[str, int]
        ) -> str:
    """Recurisve helper function to `_node_numbering`."""
    if counter not in numberwithins and counter not in counters:
        return trailing_numbering
    if counter not in numberwithins and counter in counters and trailing_numbering:
        return f'{counters[counter]}.{trailing_numbering}'
    if counter not in numberwithins and counter in counters and not trailing_numbering:
        return f'{counters[counter]}'

    parent_counter = numberwithins[counter]
    current_count = counters[counter]
    if not trailing_numbering:
        to_pass_to_trailing_numbering = str(current_count)
    else:
        to_pass_to_trailing_numbering = f'{current_count}.{trailing_numbering}'

    return _numbering_helper(
        to_pass_to_trailing_numbering,
        parent_counter,
        numberwithins,
        counters)
    

In [7800]:
#| export
# def _node_numbering(
#         node: LatexNode,
#         numbertheorem_counters: dict[str, str],
#         numberwithins: dict[str, str],
#         counters: dict[str, int]
#         ) -> str: # Just the numbering of the node, no "section/subsection" or displayname
#     if _is_section_node(node):
#         counter = 'section'
#     elif _is_subsection_node(node):
#         counter = 'subsection'
#     elif _is_environment_node(node):
#         counter = numbertheorem_counters[node.environmentname][0]
#     return _numbering_helper('', counter, numberwithins, counters)

#| export
def _node_numbering(node, numbertheorem_counters, numberwithins, counters):
    """Returns the full hierarchical number string for a node."""
    counter_name = None
    
    # 1. Identify the counter name based on node type
    if _is_section_node(node):
        counter_name = 'section'
    elif _is_subsection_node(node):
        counter_name = 'subsection'
    elif _is_subsubsection_node(node):
        counter_name = 'subsubsection'
    elif _is_environment_node(node):
        # Environments use the mapping from the preamble
        env_name = node.environmentname
        counter_name, _ = numbertheorem_counters.get(env_name, (None, None))
    
    if not counter_name:
        return ""
        
    # 2. Use the 'while' loop to climb the tree
    res = [str(counters.get(counter_name, 0))]
    curr = counter_name
    
    while curr in numberwithins:
        parent = numberwithins[curr]
        res.append(str(counters.get(parent, 0)))
        curr = parent
        
    return ".".join(reversed(res))

In [7801]:
# --- Test Environment Cases ---
# Use 'theorem' in the LaTeX to match the dictionary key
text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
node = get_node_from_simple_text(text)

# Ensure the key here matches the environment name 'theorem'
numbertheorem_counters = {'theorem': ('theorem', None)}
numberwithins = {'theorem': 'section'}
counters = {'section': 1, 'theorem': 0}

# This should now correctly return '1.0'
test_eq(_node_numbering(node, numbertheorem_counters, numberwithins, counters), '1.0')

# --- Test Counter Aliasing (thm uses equation counter) ---
# Here we map the environment 'theorem' to the counter 'equation'
numbertheorem_counters_alias = {'theorem': ('equation', None)}
counters_alias = {'section': 2, 'equation': 5}
withins_alias = {'equation': 'section'}

test_eq(_node_numbering(node, numbertheorem_counters_alias, withins_alias, counters_alias), '2.5')

# --- Test Section-like Nodes ---
text_sec = r"""\section{New section!}"""
node_sec = get_node_from_simple_text(text_sec)

# Sections don't use numbertheorem_counters, so this is safe
test_eq(_node_numbering(node_sec, numbertheorem_counters, numberwithins, counters), '1')

# --- Test Deep Nesting (The 1.1.1 Logic) ---
deep_withins = {'theorem': 'subsection', 'subsection': 'section'}
deep_counts = {'section': 1, 'subsection': 2, 'theorem': 3}
test_eq(_node_numbering(node, numbertheorem_counters, deep_withins, deep_counts), '1.2.3')

In [7802]:
# text = r"""\begin{thm}This is a theorem. \end{thm}"""
# node = get_node_from_simple_text(text)
# # Test a theoreem being counted by its own counter.
# numbertheorem_counters = {'thm': ('thm', None)}
# numberwithins = {}
# counters = {'thm': 1}
# sample_numbering = _node_numbering(
#     node, numbertheorem_counters, numberwithins, counters)
# test_eq(sample_numbering, '1')
# # Test a theorem being countered by the equation counter.
# numbertheorem_counters = {'thm': ('equation', None)}
# numberwithins = {}
# counters = {'equation': 2}
# sample_numbering = _node_numbering(
#     node, numbertheorem_counters, numberwithins, counters)
# test_eq(sample_numbering, '2')
# # Test a theorem being countered by the equation counter.
# numbertheorem_counters = {'thm': ('equation', None)}
# numberwithins = {}
# counters = {'equation': 2}
# sample_numbering = _node_numbering(
#     node, numbertheorem_counters, numberwithins, counters)
# test_eq(sample_numbering, '2')

# text = r"""\begin{corollary}This is a corollary. \end{orollary}"""
# node = get_node_from_simple_text(text)
# # Test a theorem-like environment being counted by the counter of
# # another theorem-like environment
# numbertheorem_counters = {'corollary': ('theorem', None), 'theorem': ('theorem', None)}
# numberwithins = {}
# counters = {'theorem': 0}
# sample_numbering = _node_numbering(
#     node, numbertheorem_counters, numberwithins, counters)
# test_eq(sample_numbering, '0')

# # Test a theorem-like environment whose counter is numbered within
# # The section counter.
# # First, see what happens when a theorem is called
# text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('theorem', None)}
# numberwithins = {'theorem': 'section'}
# counters = {'section': 1, 'theorem': 0}
# sample_numbering = _node_numbering(
#     node, numbertheorem_counters, numberwithins, counters)
# test_eq(sample_numbering, '1.0')

# # Next, see what happens when a new section is invoked:
# text = r"""\section{New section! The theorem counter should be reset}"""
# node = get_node_from_simple_text(text)
# sample_numbering = _node_numbering(
#     node, numbertheorem_counters, numberwithins, counters)
# test_eq(sample_numbering, '1')

# # Test a theorem-like environment sharing a counter with equation
# # and in turn equation is numbered within section.
# text = r"""\begin{theorem}This is a theorem. \end{theorem}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'theorem': ('equation', None)}
# numberwithins = {'equation': 'section'}
# counters = {'section': 1, 'equation': 0}
# sample_numbering = _node_numbering(
#     node, numbertheorem_counters, numberwithins, counters)
# test_eq(sample_numbering, '1.0')
# # Next, see what happens when a new section is invoked:
# text = r"""\section{New section! The theorem counter should be reset}"""
# node = get_node_from_simple_text(text)
# sample_numbering = _node_numbering(
#     node, numbertheorem_counters, numberwithins, counters)
# test_eq(sample_numbering, '1')

In [7803]:
#| export
# def _title_for_section_subsection_subsubsection_node(
#         node: LatexNode,
#         counters: dict[str, int],
#         numbered: bool
#     ):
#     _, title = _section_title(node.latex_verbatim())
#     if not numbered:
#         return title
#     if _is_section_node(node):
#         return f"{counters['section']}. {title}"
#     elif _is_subsection_node(node):
#         return f"{counters['section']}.{counters['subsection']}. {title}"
#     else: # _is_subsubsection_node(node):
#         return f"{counters['section']}.{counters['subsection']}.{counters['subsubsection']}. {title}"



In [7804]:
#| export
def _build_number_string(counter_name: str, counters: dict, numberwithins: dict) -> str:
    """Climbs the numberwithins hierarchy to build strings like '1.1.1'"""
    res = [str(counters.get(counter_name, 0))]
    curr = counter_name
    while curr in numberwithins:
        parent = numberwithins[curr]
        res.append(str(counters.get(parent, 0)))
        curr = parent
    return ".".join(reversed(res))

In [7805]:
#| hide
from fastcore.test import test_eq

# Mock data structures
mock_counters = {'section': 1, 'subsection': 2, 'theorem': 3}
mock_numberwithins = {'theorem': 'subsection', 'subsection': 'section'}

# Test top-level
test_eq(_build_number_string('section', mock_counters, mock_numberwithins), "1")

# Test mid-level
test_eq(_build_number_string('subsection', mock_counters, mock_numberwithins), "1.2")

# Test deep-level (The 1.1.1 fix)
test_eq(_build_number_string('theorem', mock_counters, mock_numberwithins), "1.2.3")

# Test with a missing parent (should still return the current counter)
test_eq(_build_number_string('equation', {'equation': 5}, {}), "5")

In [7806]:
#| export
# def _title_for_section_subsection_subsubsection_node(
#         node: LatexNode,
#         counters: dict[str, int],
#         numbered: bool,
#         numberwithins: dict[str, str] # Add this to the arguments
#     ):
#     _, title = _section_title(node.latex_verbatim())
#     if not numbered:
#         return title
        
#     # Determine the starting level
#     if _is_section_node(node): curr = 'section'
#     elif _is_subsection_node(node): curr = 'subsection'
#     else: curr = 'subsubsection'
    
#     # Dynamically build the number chain (e.g., 1.1.2)
#     numbering = _build_number_string(curr, counters, numberwithins)
#     return f"{numbering}. {title}"

In [7807]:
#| export
def _build_number_string(curr, counters, numberwithins):
    """Recursive helper to climb from subsubsection -> subsection -> section -> part."""
    current_val = counters.get(curr, 0)
    parent = numberwithins.get(curr)
    
    if parent and parent in counters:
        prefix = _build_number_string(parent, counters, numberwithins)
        return f"{prefix}.{current_val}"
    
    return str(current_val)

In [7808]:
#| export
def _title_for_section_subsection_subsubsection_node(
        node: LatexNode,
        counters: dict[str, int],
        numbered: bool,
        numberwithins: dict[str, str]):
    
    is_numbered, title = _section_title(node.latex_verbatim())
    if not numbered:
        return title
        
    if _is_part_node(node): curr = 'part'
    elif _is_section_node(node): curr = 'section'
    elif _is_subsection_node(node): curr = 'subsection'
    else: curr = 'subsubsection'
    
    # Use the safe formatter that ignores 0-valued parents
    numbering = get_formatted_number(curr, counters, numberwithins, {})
    return f"{numbering}. {title}"
# def _title_for_section_subsection_subsubsection_node(
#         node: LatexNode,
#         counters: dict[str, int],
#         numbered: bool,
#         numberwithins: dict[str, str]
#     ):
#     # Extract the text title (e.g., 'Algebra' or 'Modules')
#     _, title = _section_title(node.latex_verbatim())
    
#     if not numbered:
#         return title
        
#     # 1. Determine the current level
#     if _is_part_node(node):
#         curr = 'part'
#     elif _is_section_node(node):
#         curr = 'section'
#     elif _is_subsection_node(node):
#         curr = 'subsection'
#     else:
#         curr = 'subsubsection'
    
#     # 2. Build the numbering string (e.g., "1" for part, "1.1" for section)
#     # This uses the same recursive logic used for environments
#     numbering = _build_number_string(curr, counters, numberwithins)
    
#     return f"{numbering}. {title}"


In [7809]:
#| hide
from fastcore.test import test_eq
from types import SimpleNamespace
from pylatexenc.latexwalker import LatexMacroNode

# 1. Create a more complete Mock Node
mock_node = SimpleNamespace(
    macroname='subsection',
    # We add the method that _is_section_node is looking for
    isNodeType=lambda t: t == LatexMacroNode,
    # Your original code uses node.latex_verbatim() as a method
    latex_verbatim=lambda: '\\subsection{Introduction}'
)

# 2. Setup state
counts = {'section': 2, 'subsection': 3}
withins = {'subsection': 'section'}

# 3. Run Test
# This should now pass through _is_section_node and _is_subsection_node successfully
title = _title_for_section_subsection_subsubsection_node(mock_node, counts, True, withins)

test_eq(title, "2.3. Introduction")

# Test unnumbered
title_unmarked = _title_for_section_subsection_subsubsection_node(mock_node, counts, False, withins)
test_eq(title_unmarked, "Introduction")

In [7810]:
#| export

# def _title_for_environment_node(
#         node: LatexNode,
#         numbertheorem_counters: dict[str, str],
#         numberwithins: dict[str, list[str]],
#         display_names: dict[str, str],
#         counters: dict[str, int],
#         swap_numbers: bool):
#     """Return the title of an environment node.
#     If the node is not that of an theorem-like environment, then 
    
#     """
#     env_name = getattr(node, 'environmentname', getattr(node, 'environment_name', None))
#     numbered = _is_numbered(node, numbertheorem_counters)

#     # If it's not a theorem-like environment (not numbered)
#     if not numbered:
#         # Return the exact display name if mapped, otherwise the raw env_name
#         # This keeps 'abstract' as 'abstract'
#         return display_names.get(env_name, env_name)
    
#     # For numbered theorem-like environments
#     numbering = get_formatted_number(
#         env_name, counters, numberwithins, numbertheorem_counters
#     )
    
#     # Here we use capitalize() for things like 'remark' -> 'Remark'
#     display_name = display_names.get(env_name, env_name.capitalize())
    
#     if swap_numbers:
#         # Match '2.1. Theorem.'
#         return f'{numbering}. {display_name}.'
#     else:
#         # Match 'Theorem 2.1.'
#         return f'{display_name} {numbering}.'
    
#     # numbered = _is_numbered(node, numbertheorem_counters)
#     # # TODO: see what happens when environments are numbered within
#     # # sections vs. subsections
#     # if not numbered:
#     #     numbering = None
#     # else:
#     #     numbering = _node_numbering(
#     #         node, numbertheorem_counters, numberwithins, counters)
    
#     # environment = node.environmentname
#     # if environment in display_names:
#     #     display_name = display_names[environment]
#     # else:
#     #     display_name = environment
#     # if not numbered:
#     #     return display_name
#     # elif swap_numbers:
#     #     return f'{numbering}. {display_name}.'
#     # else:
#     #     return f'{display_name} {numbering}.'
        

In [7811]:
#| export
def get_formatted_number(
        env_or_counter: str, 
        counters: dict[str, int], 
        numberwithins: dict[str, str], 
        numbertheorem_counters: dict[str, tuple[str, str]]) -> str:
    """Recursively builds the numbering string, omitting zero-valued parents."""
    # Determine the actual counter name
    if env_or_counter in numbertheorem_counters:
        counter_name, _ = numbertheorem_counters[env_or_counter]
    else:
        counter_name = env_or_counter

    current_val = counters.get(counter_name, 0)
    parent = numberwithins.get(counter_name)

    # CRITICAL FIX: Only prepend the parent if it exists AND has been used (val > 0).
    # This prevents "0.1. Introduction" in documents that don't have parts.
    if parent and parent in counters and counters[parent] > 0:
        # Recursive call to get the parent's prefix
        parent_prefix = get_formatted_number(
            parent, counters, numberwithins, numbertheorem_counters)
        return f"{parent_prefix}.{current_val}"
    
    return str(current_val)

In [7812]:
#| hide
from fastcore.test import test_eq

def test_formatted_number_zero_parent():
    # Setup: Section is within Part, Equation is within Section.
    numberwithins = {
        'section': 'part',
        'subsection': 'section',
        'equation': 'section'
    }
    
    # numbertheorem_counters maps environment names to their (counter_name, reset_at)
    # In this case, we just need the mapping for 'remark' -> 'equation'
    number_thm_counters = {
        'remark': ('equation', 'section')
    }

    # Scenario A: The "Gross" case (Example 1)
    # Part exists in hierarchy but has not been used (value 0)
    counters_no_part = {
        'part': 0,
        'section': 1,
        'equation': 1
    }

    # 1. Test Section Numbering (Should be "1", NOT "0.1")
    res_sec = get_formatted_number('section', counters_no_part, numberwithins, number_thm_counters)
    print(f"Section (No Part) result: {res_sec}")
    test_eq(res_sec, "1")

    # 2. Test Remark Numbering (Should be "1.1", NOT "0.1.1")
    res_rem = get_formatted_number('remark', counters_no_part, numberwithins, number_thm_counters)
    print(f"Remark (No Part) result: {res_rem}")
    test_eq(res_rem, "1.1")

    # Scenario B: The "Example 14" case
    # Part HAS been used (value 1)
    counters_with_part = {
        'part': 1,
        'section': 1,
        'equation': 1
    }

    # 3. Test Section Numbering (Should be "1.1")
    res_sec_p = get_formatted_number('section', counters_with_part, numberwithins, number_thm_counters)
    print(f"Section (With Part) result: {res_sec_p}")
    test_eq(res_sec_p, "1.1")

    # 4. Test Remark Numbering (Should be "1.1.1")
    res_rem_p = get_formatted_number('remark', counters_with_part, numberwithins, number_thm_counters)
    print(f"Remark (With Part) result: {res_rem_p}")
    test_eq(res_rem_p, "1.1.1")

test_formatted_number_zero_parent()

Section (No Part) result: 1
Remark (No Part) result: 1.1
Section (With Part) result: 1.1
Remark (With Part) result: 1.1.1


In [7813]:
#| export
# def _title_for_environment_node(
#         node: LatexNode, 
#         numbertheorem_counters: dict[str, tuple[str, str]], 
#         numberwithins: dict[str, str], 
#         display_names: dict[str, str], 
#         counters: dict[str, int], 
#         swap_numbers: bool) -> str:
#     """Builds the title for an environment like Theorem or Remark."""
#     env_name = node.environmentname
    
#     # Check if the environment is a known theorem-like environment
#     is_thm_like = env_name in numbertheorem_counters
    
#     # If it's theorem-like but not in display_names, capitalize it (e.g., 'remark' -> 'Remark')
#     # If it's NOT theorem-like (e.g., 'abstract'), keep it raw to pass the 'abstract' test.
#     if is_thm_like:
#         display_name = display_names.get(env_name, env_name.capitalize())
#     else:
#         display_name = display_names.get(env_name, env_name)
    
#     if not _is_numbered(node, numbertheorem_counters):
#         return display_name

#     num_str = get_formatted_number(env_name, counters, numberwithins, numbertheorem_counters)
    
#     if swap_numbers:
#         return f"{num_str}. {display_name}."
#     else:
#         return f"{display_name} {num_str}."

In [7814]:
#| export
#| export
def _title_for_environment_node(
        node: LatexEnvironmentNode,
        numbertheorem_counters: dict[str, tuple[str, Union[str, None]]],
        numberwithins: dict[str, str],
        display_names: dict[str, str],
        counters: dict[str, int],
        swap_numbers: bool = False
        ) -> str:
    env_name = node.environmentname
    if env_name not in numbertheorem_counters:
        return display_names.get(env_name, env_name)

    display_name = display_names.get(env_name, env_name.capitalize())
    numbering = get_formatted_number(
        env_name, counters, numberwithins, numbertheorem_counters)

    if swap_numbers:
        return f"{numbering}. {display_name}."
    else:
        return f"{display_name} {numbering}."

In [7815]:
#| hide
def test_environment_titles():
    class MockNode:
        def __init__(self, name): self.environmentname = name
        def isNodeType(self, t): return t == LatexEnvironmentNode

    numbertheorem_counters = {'theorem': ('theorem', None)}
    numberwithins = {}
    display_names = {'theorem': 'Theorem'}
    counters = {'theorem': 1}

    # 1. Numbered Theorem
    node_thm = MockNode('theorem')
    title_thm = _title_for_environment_node(node_thm, numbertheorem_counters, numberwithins, display_names, counters)
    print(f"Numbered Title: {title_thm}")
    assert title_thm == "Theorem 1."

    # 2. Unnumbered Environment (e.g., abstract) - Should match exact case if not in display_names
    node_abs = MockNode('abstract')
    title_abs = _title_for_environment_node(node_abs, numbertheorem_counters, numberwithins, display_names, counters)
    print(f"Unnumbered Title (abstract): {title_abs}")
    assert title_abs == "abstract"

    # 3. Swapped Numbers (e.g., 1. Theorem.)
    title_swap = _title_for_environment_node(node_thm, numbertheorem_counters, numberwithins, display_names, counters, swap_numbers=True)
    print(f"Swapped Title: {title_swap}")
    assert title_swap == "1. Theorem."

test_environment_titles()

Numbered Title: Theorem 1.
Unnumbered Title (abstract): abstract
Swapped Title: 1. Theorem.


In [7816]:
#| hide

def test_title_formatting_hierarchy():
    # Setup standard counters and display names
    display_names = {'remark': 'Remark', 'theorem': 'Theorem', 'section': 'Section'}
    numbertheorem_counters = {
        'remark': ('equation', 'section'), 
        'theorem': ('theorem', 'subsection')
    }
    numberwithins = {
        'equation': 'section',
        'subsection': 'section',
        'theorem': 'subsection'
    }

    # Robust Mock for pylatexenc LatexNode
    class MockNode:
        def __init__(self, name, node_type): 
            self.environmentname = name if node_type == 'environment' else None
            self.macroname = name if node_type == 'macro' else None
            self.node_type = node_type
            
        def isNodeType(self, t):
            # Simulate pylatexenc types
            if self.node_type == 'environment': return 'LatexEnvironmentNode' in str(t)
            if self.node_type == 'macro': return 'LatexMacroNode' in str(t)
            return False

    # Case 1: Remark -> Equation -> Section (The "Remark 1.1" fix)
    counters_1 = {'section': 1, 'equation': 1}
    # Note: 'remark' is an environment
    remark_node = MockNode('remark', 'environment')
    
    title_1 = _title_for_environment_node(
        remark_node, numbertheorem_counters, numberwithins, 
        display_names, counters_1, swap_numbers=False
    )
    test_eq(title_1, 'Remark 1.1.')

    # Case 2: Deep nesting (Theorem 2.1.1)
    counters_2 = {'section': 2, 'subsection': 1, 'theorem': 1}
    thm_node = MockNode('theorem', 'environment')
    
    title_2 = _title_for_environment_node(
        thm_node, numbertheorem_counters, numberwithins, 
        display_names, counters_2, swap_numbers=False
    )
    test_eq(title_2, 'Theorem 2.1.1.')

test_title_formatting_hierarchy()

In [7817]:
#| hide
from pylatexenc.latexwalker import LatexMacroNode, LatexEnvironmentNode

def test_title_logic_comprehensive():
    # Setup mocks and data
    display_names = {'remark': 'Remark', 'theorem': 'Theorem', 'proof': 'Proof'}
    numbertheorem_counters = {
        'remark': ('equation', 'section'), 
        'theorem': ('theorem', 'subsection')
    }
    numberwithins = {
        'equation': 'section',
        'subsection': 'section',
        'theorem': 'subsection'
    }

    class MockNode:
        def __init__(self, name, node_type, verbatim=""): 
            self.environmentname = name if node_type == LatexEnvironmentNode else None
            self.macroname = name if node_type == LatexMacroNode else None
            self._type = node_type
            self._verbatim = verbatim
            
        def isNodeType(self, t): return self._type == t
        def latex_verbatim(self): return self._verbatim

    # Case 1: The "Remark 1.1" fix (Environment -> Counter -> Parent)
    counters_1 = {'section': 1, 'equation': 1}
    remark_node = MockNode('remark', LatexEnvironmentNode)
    
    title_1 = _title_for_environment_node(
        remark_node, numbertheorem_counters, numberwithins, 
        display_names, counters_1, swap_numbers=False
    )
    test_eq(title_1, 'Remark 1.1.')

    # Case 2: Deep Nesting (Theorem 2.1.1)
    # Section 2, Subsection 1, Theorem 1
    counters_2 = {'section': 2, 'subsection': 1, 'theorem': 1}
    thm_node = MockNode('theorem', LatexEnvironmentNode)
    
    title_2 = _title_for_environment_node(
        thm_node, numbertheorem_counters, numberwithins, 
        display_names, counters_2, swap_numbers=False
    )
    test_eq(title_2, 'Theorem 2.1.1.')

    # Case 3: Unnumbered Environment (Proof)
    # _is_numbered should return False if not in numbertheorem_counters
    proof_node = MockNode('proof', LatexEnvironmentNode)
    title_3 = _title_for_environment_node(
        proof_node, numbertheorem_counters, numberwithins, 
        display_names, counters_1, swap_numbers=False
    )
    test_eq(title_3, 'Proof')

    # Case 4: Swap Numbers (1.1. Remark.)
    title_4 = _title_for_environment_node(
        remark_node, numbertheorem_counters, numberwithins, 
        display_names, counters_1, swap_numbers=True
    )
    test_eq(title_4, '1.1. Remark.')

test_title_logic_comprehensive()

In [7818]:
#| export

def _title(
        node: LatexNode,
        numbertheorem_counters: dict[str, str],
        numberwithins: dict[str, str], # An output of _setup_numberwithins
        all_numberwithins: dict[str, list[str]], # An output of all_numberwithins
        display_names: dict[str, str],
        counters: dict[str, int],
        swap_numbers: bool):
    """Return the title of a node based on the count in
    `counters`"""
    numbered = _is_numbered(node, numbertheorem_counters)

    if (_is_section_node(node) or _is_subsection_node(node)
            or _is_subsubsection_node(node)):
            # Inside _title
        # return _title_for_section_subsection_subsubsection_node(
        #     node, counters, bool)
        return _title_for_section_subsection_subsubsection_node(
            node, counters, numbered, numberwithins) # Pass numberwithins here!
    # if _is_section_node(node) and numbered:
    #     _, title = _section_title(node.latex_verbatim())
    #     return f"{counters['section']}. {title}"
    # if _is_section_node(node) and not numbered:
    #     _, title = _section_title(node.latex_verbatim())
    #     return title 

    # if _is_subsection_node(node) and numbered:
    #     _, title = _section_title(node.latex_verbatim())
    #     return f"{counters['section']}.{counters['subsection']}. {title}"
    # if _is_subsection_node(node) and not numbered:
    #     _, title = _section_title(node.latex_verbatim())
    #     return title

    if _is_environment_node(node):
        return _title_for_environment_node(
            node, numbertheorem_counters, numberwithins,
            display_names, counters, swap_numbers)



In [7819]:
#| hide

# Theorem that is not numbered within anything
text = r"""\begin{thm}This is a theorem. \end{thm}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'thm': ('thm', None)}
numberwithins = {}
all_numberwithins = {}
display_names = {'thm': 'Theorem'}
counters = {'thm': 1}
swap_numbers = False
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, 'Theorem 1.')

swap_numbers = False
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, 'Theorem 1.')

# Theorem that is counted by equation, which in turn is numbered within
# section
text = r"""\begin{thm}This is a theorem. \end{thm}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'thm': ('equation', None)}
numberwithins = {'equation': 'section'}
all_numberwithins = {'equation': ['section']}
display_names = {'thm': 'Theorem'}
counters = {'equation': 1, 'section': 2}
swap_numbers = False
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, 'Theorem 2.1.')

swap_numbers = True
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, '2.1. Theorem.')

# Section
text = r"""\section{This is a section}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'thm': ('equation', None)}
numberwithins = {'equation': 'section'}
all_numberwithins = {'equation': ['section']}
display_names = {'thm': 'Theorem'}
counters = {'equation': 1, 'section': 2}
swap_numbers = False
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, '2. This is a section')

swap_numbers = True
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, '2. This is a section')

# Subsection
text = r"""\subsection{This is a subsection}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'thm': ('equation', None)}
numberwithins = {'equation': 'section', 'subsection': 'section'}
all_numberwithins = {'equation': ['section'], 'subsection': ['section']}
display_names = {'thm': 'Theorem'}
counters = {'equation': 1, 'section': 2, 'subsection': 3}
swap_numbers = False
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, '2.3. This is a subsection')

swap_numbers = True
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, '2.3. This is a subsection')

# In the case that an environment is not a theorem-like environment.
text = r"""\begin{abstract} This is an abstract \end{abstract}"""
node = get_node_from_simple_text(text)
numbertheorem_counters = {'thm': ('equation', None)}
numberwithins = {'equation': 'section', 'subsection': 'section'}
all_numberwithins = {'equation': ['section'], 'subsection': ['section']}
display_names = {'thm': 'Theorem'}
counters = {'equation': 1, 'section': 2, 'subsection': 3}
swap_numbers = False
sample_title = _title(
    node, numbertheorem_counters, numberwithins, all_numberwithins,
    display_names, counters, swap_numbers)
test_eq(sample_title, 'abstract')

# # In the case a section has multilines
# text = r"""\section{Exceptional maximal subgroups of 
# \texorpdfstring{\(\GSp_4(\ff_\ell)\)}{GSp4Fell}}"""
# node = get_node_from_simple_text(text)
# numbertheorem_counters = {'thm': ('equation', None)}
# numberwithins = {'equation': 'section', 'subsection': 'section'}
# all_numberwithins = {'equation': ['section'], 'subsection': ['section']}
# display_names = {'thm': 'Theorem'}
# counters = {'equation': 1, 'section': 2, 'subsection': 3}
# swap_numbers = False
# sample_title = _title(
#     node, numbertheorem_counters, numberwithins, all_numberwithins,
#     display_names, counters, swap_numbers)
# sample_title
# test_eq(sample_title, '2.3. This is a subsection')

In [7820]:
#| export
def swap_numbers_invoked(
        preamble: str
        ) -> bool: # 
    r"""Returns `True` if `\swapnumbers` is in the preamble.

    Assume that a mention of `\swapnumbers` is an actual invocation.
    """
    preamble = remove_comments(preamble)
    return r'\swapnumbers' in preamble

In [7821]:
assert swap_numbers_invoked(r'\swapnumbers')
assert not swap_numbers_invoked(r'''
\documentclass{article}
\usepackage{amsthm}
%\usepackage{amsmath}

\newtheorem{theorem}{Theorem} % \swapnumbers
\newtheorem{corollary}[theorem]{Corollary}
\newtheorem{definition}[theorem]{Definition}
\newtheorem*{remark*}{Remark}''')

In [7822]:
#| export
def _node_warrants_own_part(
        node, environments_to_not_divide_along: list[str],
        accumulation: str, parts: list[tuple[str, str]]) -> bool:
    """Return `True` if `node` warrants making a new part to be added in `parts`.

    This is a helper function for `_process_node`. When this function returns
    `True`, the `accumulation` should be considered for appending to `parts`
    and the node should also be appended to `parts
    """
    if _is_section_node(node) or _is_subsection_node(node) or _is_subsubsection_node(node):
        return True
    elif not _is_environment_node(node):
        return False
    # Is environment node from here and below.
    if len(parts) == 0 and accumulation.strip() == '':
        return True
    return node.environmentname not in environments_to_not_divide_along

In [7823]:
#| hide

# These examples are based on `numbering_example_1_consecutive_numbering_scheme`
# in `\nbs\_tests\latex_examples`.

# Test the case of accumulating text at the very beginning before any section
node = get_node_from_simple_text('\nFor this document, the `theorem` counter is not reset whenever a new section begins.\n\nA similar numbering scheme can be accomplished by importing ')
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*']
accumulation = ''
parts = []
assert not _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

node = get_node_from_simple_text('\\verb|amsmath|')
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*']
accumulation = '\nFor this document, the `theorem` counter is not reset whenever a new section begins.\n\nA similar numbering scheme can be accomplished by importing '
parts = []
assert not _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

node = get_node_from_simple_text(' and invoking the code ')
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*']
accumulation = '\nFor this document, the `theorem` counter is not reset whenever a new section begins.\n\nA similar numbering scheme can be accomplished by importing \\verb|amsmath|'
parts = []
assert not _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

# Now a new section comes in, which warrants a new part.
node = get_node_from_simple_text('\\section{Introduction}')
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*']
accumulation = '\nFor this document, the `theorem` counter is not reset whenever a new section begins.\n\nA similar numbering scheme can be accomplished by importing \\verb|amsmath| and invoking the code \\verb|\\numberwithin{theorem}{part}| in the preamble.\n\n' 
parts = []
assert _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

# Now a new theorem comes in, which also warrants a new part.
node = get_node_from_simple_text('\\begin{theorem}\nThis is Theorem 1.\n\\end{theorem}')
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*']
accumulation = '\n\n' 
parts = [['1', 'For this document, the `theorem` counter is not reset whenever a new section begins.\n\nA similar numbering scheme can be accomplished by importing \\verb|amsmath| and invoking the code \\verb|\\numberwithin{theorem}{part}| in the preamble.'], ['1. Introduction', '\\section{Introduction}']] 
assert _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

# Test the case where text that does not belong to an envrionment occurs at the 
# very beginning, even before any sections. and an environment node makes an
# appearance.
# cf. divide_latex_example_2 in `nbs\_tests\latex_examples`.
node = get_node_from_simple_text(r"""\begin{abstract}
    This is an abstract
    \end{abstract}""")
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*']
accumulation = r'\maketitle\n'
parts = []
assert _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

# Now test the same case except `abstract` is included in `environments_to_not_divide_along`
node = get_node_from_simple_text(r"""\begin{abstract}
    This is an abstract
    \end{abstract}""")
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*', 'abstract']
accumulation = r'\maketitle\n'
parts = []
assert not _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

# Test the case at the beginning of a section with an enumerate node.
node = get_node_from_simple_text('\\begin{enumerate}\n  \\item Introduction 2\n\n  \\item Preliminaries $\\quad 7$\n\n\\end{enumerate}')
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*', 'enumerate', 'itemize']
accumulation = r'\n'
parts = [['1. CONTENTS', '\\section{CONTENTS}']]
assert not _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

# Test the case where a section is immediately followed by a subsection
node = get_node_from_simple_text('\\section{Section 2}')
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*', 'enumerate', 'itemize']
accumulation = r'\n'
parts = [['1. Section 1', '\\section{Section 1}']]
assert _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

# Test the case of a subsubsection
node = get_node_from_simple_text('\\subsubsection{section 1.1.1}')
environments_to_not_divide_along = ['equation', 'equation*', 'proof', 'align', 'align*', 'enumerate', 'itemize']
accumulation = r'\n'
parts = [['1.1 Section 1.1', '\\subsection{Section 1.1}']]
assert _node_warrants_own_part(node, environments_to_not_divide_along, accumulation, parts)

In [7824]:
#| export
def _node_is_proof_immediately_following_a_theorem_like_environment(
        node, accumulation, parts, display_names) -> bool:
    """Return `True` if `node` is that of a proof environment that immediately
    follows a theorem-like environment.
    
    This is a helper function for `_process_node`.
    """
    if not _is_environment_node(node):
        return False
    if not node.environmentname == 'proof':
        return False
    if not len(parts) > 0:
        return False
    if accumulation.strip() != '':
        return False
    if not _text_is_of_environment_node(parts[-1]['text']):
        return False
    return _environment_name_of_text(parts[-1]['text']) in display_names
    # previous_node = get_node_from_simple_text(parts[-1][1])
    # if not _is_environment_node(previous_node):
    #     return False
    # return previous_node.environmentname in display_names


def _node_is_nonspecial_following_a_sectionlike_node(
        node, environments_to_not_divide_along, accumulation, parts) -> bool:
    """Return `True` if `node` is that of a non-environment and non-setionlike
    node that immediately follows a sectionlike (section, subsection,
    or subsubsection) node.
    
    This is a helper function for `_process_node`.
    """
    if ((_is_environment_node(node) and node.environmentname not in environments_to_not_divide_along)
            or _is_section_node(node)
            or _is_subsection_node(node)
            or _is_subsubsection_node(node)):
        return False
    if len(parts) == 0:
        return False
    if accumulation.strip() != '':
        return False
    return _text_is_of_section_like_node(parts[-1]['text'])
    # return _text_is_of_section_like_node(parts[-1]["text"])
    

In [7825]:
#| hide

# Test basic case where node is proof node following a theorem node.
node = get_node_from_simple_text(r'\begin{proof} This is a proof \end{proof}')
accumulation = '\n\n'
# parts = [['1. Section', '\\section{Section}'], ['Theorem 1.', r'\begin{thm} This is a theorem \end{thm}']]
parts = [DividedLatexPart(note_title='1. Section', text='\\section{Section}'), DividedLatexPart(note_title='Theorem 1.', text=r'\begin{thm} This is a theorem \end{thm}')]
display_names = {'thm': 1}
assert _node_is_proof_immediately_following_a_theorem_like_environment(node, accumulation, parts, display_names)

# Test basic case where node is not a proof node
node = get_node_from_simple_text(r'\\begin{thm} This is a theorem \end{thm}')
accumulation = '\n\n'
# parts = [['1. Section', '\\section{Section}']]
parts = [DividedLatexPart(note_title='1. Section', text='\\section{Section}')]
display_names = {'thm': 0}
assert not _node_is_proof_immediately_following_a_theorem_like_environment(node, accumulation, parts, display_names)

# Test when node is proof node at the very beginning of a document.
node = get_node_from_simple_text(r'\begin{proof} This is a proof \end{proof}')
accumulation = '\n\n'
parts = []
display_names = {'thm': 0}
assert not _node_is_proof_immediately_following_a_theorem_like_environment(node, accumulation, parts, display_names)

# Test when node is proof node at the beginnning of a section.
node = get_node_from_simple_text(r'\begin{proof} This is a proof \end{proof}')
accumulation = '\n\n'
# parts = [['1. Section', '\\section{Section}']]
parts = [DividedLatexPart(note_title='1. Section', text='\\section{Section}')]
display_names = {'thm': 0}
assert not _node_is_proof_immediately_following_a_theorem_like_environment(node, accumulation, parts, display_names)

# Test when node is proof node following a remark.
node = get_node_from_simple_text(r'\begin{proof} This is a proof \end{proof}')
accumulation = '\n\n'
# parts = [['1. Section', '\\section{Section}'], ['Theorem 1.', r'\\begin{thm} This is a theorem \end{thm}'], ['Remark', r'\\begin{rem} This is an unnumbered remark \\end{rem}']]
parts = [
    DividedLatexPart(note_title='1. Section', text='\\section{Section}'),
    DividedLatexPart(note_title='Theorem 1.', text=r'\\begin{thm} This is a theorem \end{thm}'),
    DividedLatexPart(note_title='Remark', text=r'\\begin{rem} This is an unnumbered remark \\end{rem}')]
display_names = {'thm': 1}
assert not _node_is_proof_immediately_following_a_theorem_like_environment(node, accumulation, parts, display_names)

# Test when node is proof node following some nonempty text
node = get_node_from_simple_text(r'\begin{proof} This is a proof \end{proof}')
accumulation = '\n\nSome things are being said before the proof but after the theorem.'
# parts = [['1. Section', '\\section{Section}'], ['Theorem 1.', r'\\begin{thm} This is a theorem \end{thm}']]
parts = [
    DividedLatexPart(note_title='1. Section', text='\\section{Section}'),
    DividedLatexPart(note_title='Theorem 1.', text=r'\\begin{thm} This is a theorem \end{thm}'),]
display_names = {'thm': 1}
assert not _node_is_proof_immediately_following_a_theorem_like_environment(node, accumulation, parts, display_names)

# Test when node is some "normal" node following some section-like node
node = get_node_from_simple_text(r'I am just some text. Not an environment, not a section')
accumulation = ''
# parts = [['1. Section', '\\section{Section}'], ]
parts = [
    DividedLatexPart(note_title='1. Section', text='\\section{Section}'),]
environments_to_not_divide_along = [
    'displaymath', 'displaymath*', 'equation', 'equation*', 'gather', 'gather*', 'multiline', 'multiline*',
    'proof', 'align', 'align*', 'enumerate', 'itemize', 'label', 'eqnarray', 'quote', 'tabular', 'table']

assert _node_is_nonspecial_following_a_sectionlike_node(node, environments_to_not_divide_along, accumulation, parts)

# Test when node is some "normal" node following some other "normal" text following a section-like node
node = get_node_from_simple_text(r'I am just some text. Not an environment, not a section')
accumulation = 'But there is some other preceding text between the start of the section and the text, so False should be returned'
# parts = [['1. Section', '\\section{Section}'], ]
parts = [
    DividedLatexPart(note_title='1. Section', text='\\section{Section}'),]
environments_to_not_divide_along = [
    'displaymath', 'displaymath*', 'equation', 'equation*', 'gather', 'gather*', 'multiline', 'multiline*',
    'proof', 'align', 'align*', 'enumerate', 'itemize', 'label', 'eqnarray', 'quote', 'tabular', 'table']

assert not _node_is_nonspecial_following_a_sectionlike_node(node, environments_to_not_divide_along, accumulation, parts)

# Test when node is a sectionlike node following a section-like node:
node = get_node_from_simple_text(r'\section{Next section}')
accumulation = ''
# parts = [['1. Section', '\\section{Section}'], ]
parts = [
    DividedLatexPart(note_title='1. Section', text='\\section{Section}'),]
environments_to_not_divide_along = [
    'displaymath', 'displaymath*', 'equation', 'equation*', 'gather', 'gather*', 'multiline', 'multiline*',
    'proof', 'align', 'align*', 'enumerate', 'itemize', 'label', 'eqnarray', 'quote', 'tabular', 'table']
assert not _node_is_nonspecial_following_a_sectionlike_node(node, environments_to_not_divide_along, accumulation, parts)

In [7826]:
#| export
def _process_node(
        node,
        environments_to_not_divide_along: list[str],
        accumulation: str,
        numbertheorem_counters: dict[str, tuple[str, Union[str, None]]],
        numberwithins: dict[str, str],
        all_numberwithins: dict[str, list[str]],
        counters: dict[str, int],
        display_names: dict[str, str],
        swap_numbers: bool,
        parts: list[DividedLatexPart]
        ) -> str:
    """
    Update `accumulation`, `counter`, and `parts` based on the contents of `node`.

    Also return 'accumulation` to update it.

    This is a helper function for `divide_latex_text`.

    """
    # If node is a proof immediately following a theorem-like environment
    # Then add it to said theorem-like environment
    # _change_counters(
    #     node, counters, numbertheorem_counters, numberwithins)
    # Correct this in _process_node
    _change_counters(
        node, counters, numbertheorem_counters, all_numberwithins) # Use 'all' for resets
    if (_node_is_proof_immediately_following_a_theorem_like_environment(
            node, accumulation, parts, display_names)
        or _node_is_nonspecial_following_a_sectionlike_node(
            node, environments_to_not_divide_along, accumulation, parts)):
        # parts[-1][1] += node.latex_verbatim()
        parts[-1]['text'] += node.latex_verbatim()
    elif _node_warrants_own_part(
            node, environments_to_not_divide_along, accumulation, parts):
        accumulation =  _append_non_environment_accumulation_to_parts_if_non_empty(
            accumulation, counters, parts)
        
        title = _title(
            node, numbertheorem_counters, numberwithins, all_numberwithins,
            display_names, counters, swap_numbers).strip()
        title = title.replace('\n', '') 
        parts.append(DividedLatexPart(note_title=title, text=node.latex_verbatim()))
    else:
        accumulation += node.latex_verbatim()
        # In _change_counters`, the '' counter is incremented by default.
        # This offsets the incorrectly incrementation.
    _change_counters_antecedently(node, counters, numbertheorem_counters, all_numberwithins)
    return accumulation



In [7827]:
#| export
def _append_non_environment_accumulation_to_parts_if_non_empty(
        accumulation: str,
        counters,
        parts: list[DividedLatexPart]) -> str:
    """Append accumulation to `parts` if `accumulation` is nonempty
    and return the updated `accumulation` """
    if accumulation.strip() != '':
        counters[''] += 1
        parts.append(
            DividedLatexPart(note_title=str(counters['']).strip(), text=accumulation.strip()))
            # [str(counters['']).strip(), accumulation.strip()])
        return ''
    else:
        return accumulation.strip()


In [7828]:
#| export
DEFAULT_ENVIRONMENTS_TO_NOT_DIVIDE_ALONG = [
    'align', 'align*', 'center', 'diagram', 'displaymath', 'displaymath*', 'enumerate', 'eqnarray', 'eqnarray*',
    'equation', 'equation*', 'gather', 'gather*', 'itemize', 'label',
    'multiline', 'multiline*', 'multline', 'multline*',
    'proof', 'quote', 'tabular', 'table', ]
def divide_latex_text(
        document: str, 
        # environments_to_divide_along: list[str], # A list of the names of environments that warrant a new note
        # numbered_environments: list[str], # A list of the names of environments which are numbered in the latex code. 
        dir: Optional[PathLike], # The directory where the included files and style files are to be found.
        environments_to_not_divide_along: list[str] = DEFAULT_ENVIRONMENTS_TO_NOT_DIVIDE_ALONG, # A list of the names of the environemts along which to not make a new note, unless the environment starts a section (or the entire document).
        replace_commands_in_document_first: bool = True,  # If `True`, invoke `replace_commands_in_latex_document` on `document` to first replace custom commands (in the document minus the preamble) before starting to divide the document.
        repeat_replacing_commands: int = -1,  # If `replace_commands_in_document_first` is `True`, then this is passed as the `repeat` argument into the invocation of `replace_commands_in_latex_document`.
        ) -> list[DividedLatexPart]: # Each tuple is of the form `(note_title, text)`, where `note_title` often encapsulates the note type (i.e. section/subsection/display text of a theorem-like environment) along with the numbering and `text` is the text of the part. Sometimes `title` is just a number, which means that `text` is not of a `\section` or `\subsection` command and not of a theorem-like environment.
    r"""Divide LaTeX text to convert into Obsidian.md notes.

    Assumes that the counters in the LaTeX document are either the
    predefined ones or specified by the `\newtheorem` command.

    Proof environments are assigned to the same parts their prcededing
    theorem-like environments, if available.

    TODO: Implement counters specified by `\newcounter`, cf. 
    https://www.overleaf.com/learn/latex/Counters#LaTeX_commands_for_working_with_counters.
    """
    numbertheorem_counters: dict[str, tuple[str, Union[str, None]]] = numbered_newtheorems_counters_in_preamble(document)
    explicit_numberwithins = numberwithins_in_preamble(document)
    # numberwithins: dict[str, str] = _setup_numberwithins(explicit_numberwithins, numbertheorem_counters)
    # all_numberwithins: dict[str, list[str]] = _setup_all_numberwithins(explicit_numberwithins, numbertheorem_counters)
    # --- NEW HIERARCHICAL LOGIC ---
    # hierarchy maps child -> parent (used for string formatting)
    numberwithins = get_counter_hierarchy(document)
    
    # reset_hierarchy maps parent -> [children] (used for recursive resets)
    all_numberwithins = get_reset_hierarchy(document)
    display_names: dict[str, str] = display_names_of_environments(document)
    counters: dict[str, int] = _setup_counters(numbertheorem_counters)
    unnumbered_environments = _unnumbered_environments(
        numbertheorem_counters, display_names)
    # Eventually gets returned
    preamble, main_document = divide_preamble(document)
    preamble = replace_inclusion_of_style_file_with_code(preamble, dir)
    preamble_commands = custom_commands(preamble)
    if replace_commands_in_document_first:
        main_document = replace_input_and_include(
            main_document, dir, preamble_commands,
            repeat_replacing_commands,
            replace_spaced_commands=False)
        # main_document = replace_commands_in_latex_document(document, repeat_replacing_commands)
    document_node = find_document_node(main_document)
    swap_numbers: bool = swap_numbers_invoked(preamble)
    parts: list[DividedLatexPart] = []
    # "Accumulates" a "part" until text that should comprise a new part is encountered
    accumulation: str = '' 
    for node in document_node.nodelist:
        accumulation = _process_node(
            node, environments_to_not_divide_along, accumulation,
            numbertheorem_counters,
            numberwithins, all_numberwithins, counters,
            display_names, swap_numbers, parts)
    _append_non_environment_accumulation_to_parts_if_non_empty(
        accumulation, counters, parts)
    return parts



In [7829]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_text_with_gather_environment'
file =  dir / 'main.tex'
sample_latex_text = text_from_file(file)
# print(sample_latex_text)
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(sample_latex_text, dir)
print(parts)

[{'note_title': '1. Introduction', 'text': '\\section{Introduction}\n\nThere is an equation\n\\begin{align*}\nasdf\n\\end{align*}\nbut this equation should not get to start its own part.\n\n'}]


#### Examples for the `divide_latex_text` function

In [7830]:
#| hide
dir = _test_directory() / 'latex_examples' / 'numbering_example_6'
file =  dir / 'main.tex'
sample_latex_text = text_from_file(file)
# print(sample_latex_text)
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(sample_latex_text, dir)
print(parts)

[{'note_title': '1. Isotypicity and non-unitarity', 'text': "\\section{Isotypicity and non-unitarity}\\label{section:bilinear-pairing}\nThe main result of this section is \\autoref{theorem:isotypic}, which states that\ncounterexamples to Putman-Wieland in genus $\\geq 3$ cannot be isotypic, i.e.,\nthere exists an element of \n$H^1(\\Sigma_{g'}, \\mathbb C)^\\rho$ with infinite orbit under the action of a finite\nindex subgroup of the mapping class group. We show more, namely \\autoref{theorem:non-unitary}: if $X\\to Y$ is an $H$-cover, where $Y$ has genus at least $3$, the virtual action of the mapping class group of $Y$ on an $H$-isotypic component of the cohomology of $X$ is non-unitary.\n\nIn \\autoref{corollary:boggi-looijenga} we use this to show how a\n result from the retracted paper of Boggi-Looijenga \\cite{boggiL:curves-with-prescribed-symmetry} would imply the Putman-Wieland conjecture.\n\nOur main tool for proving this is a natural bilinear pairing, which we\nnext introduce

In the following example, we take a basic LaTeX file and divide it into parts:

In [7831]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_proof_preceded_by_theorem'
file = dir / 'main.tex'
sample_latex_text = text_from_file(file)
print(sample_latex_text)


\documentclass[10pt]{article}

\theoremstyle{plain}
\newtheorem*{theorem*}{Theorem}
\newtheorem*{theoremA}{Theorem A}
\newtheorem*{theoremB}{Theorem B}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem{proposition}[equation]{Proposition}
\newtheorem{lemma}[equation]{Lemma}
\newtheorem{corollary}[equation]{Corollary}

\theoremstyle{definition}
\newtheorem{definition}[equation]{Definition}
\newtheorem{example}[equation]{Example}
\newtheorem*{acknowledgements}{Acknowledgements}
\newtheorem*{conventions}{Conventions}

\theoremstyle{remark}
\newtheorem{remark}[equation]{Remark}



\begin{document}
\section{Some section}

\begin{theorem}
This is a theorem.
\end{theorem}

\begin{proof}
This is a proof
\end{proof}

\end{document}


The `divide_preamble` function recognizes where the preamble ends and where the document begins.

In [7832]:
preamble, document = divide_preamble(sample_latex_text)

In [7833]:
print(preamble)


\documentclass[10pt]{article}

\theoremstyle{plain}
\newtheorem*{theorem*}{Theorem}
\newtheorem*{theoremA}{Theorem A}
\newtheorem*{theoremB}{Theorem B}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem{proposition}[equation]{Proposition}
\newtheorem{lemma}[equation]{Lemma}
\newtheorem{corollary}[equation]{Corollary}

\theoremstyle{definition}
\newtheorem{definition}[equation]{Definition}
\newtheorem{example}[equation]{Example}
\newtheorem*{acknowledgements}{Acknowledgements}
\newtheorem*{conventions}{Conventions}

\theoremstyle{remark}
\newtheorem{remark}[equation]{Remark}






In [7834]:
print(document)

\begin{document}
\section{Some section}

\begin{theorem}
This is a theorem.
\end{theorem}

\begin{proof}
This is a proof
\end{proof}

\end{document}


The `divide_latex_text` function divides the LaTeX document into parts, generally based on setions and theorem-like environments:

In [7835]:

parts = divide_latex_text(sample_latex_text, dir)
print(parts)
test_eq(len(parts), 2)

[{'note_title': '1. Some section', 'text': '\\section{Some section}\n\n'}, {'note_title': 'Theorem 1.1.', 'text': '\\begin{theorem}\nThis is a theorem.\n\\end{theorem}\\begin{proof}\nThis is a proof\n\\end{proof}'}]


In the next example, we have some `enumerate` environments to list out some things. The `divide_latex_text` does not create a new part of the `enumerate` environment.

In [7836]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_text_preceded_by_undivided_environment' /_test_directory() / 'latex_examples' / 'divide_latex_example_text_preceded_by_undivided_environment'
file = dir / 'main.tex'
sample_latex_text = text_from_file(file)
print(sample_latex_text)

% In this example, there are enumerate environments, which should not get their
% own `part`, cf. `divide_latex_text` in `16_latex.convert.ipynb`.
\documentclass[10pt]{article}
\usepackage{amsmath}
\usepackage{amsfonts}
\begin{document}

\section{Introduction}

Blahblahblah, this document has some lists.
The `divide_latex_text` should not create a separate part for the below `enumerate`
environment; after all, it seems better to include the list in the same file/note
as the text that provides context for the list.

\begin{enumerate}
  \item Rings
  \item Fields
\end{enumerate}

And here is another list, perhaps a grocery list:

\begin{enumerate}
  \setcounter{enumi}{3}
  \item apples
  \item bananas
  \item milk
\end{enumerate}

Lalalala

\end{document}


In [7837]:
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(document, dir)
print(parts)
test_eq(len(parts), 1)

[{'note_title': '1. Introduction', 'text': '\\section{Introduction}\n\nBlahblahblah, this document has some lists.\nThe `divide_latex_text` should not create a separate part for the below `enumerate`\nenvironment; after all, it seems better to include the list in the same file/note\nas the text that provides context for the list.\n\n\\begin{enumerate}\n  \\item Rings\n  \\item Fields\n\\end{enumerate}\n\nAnd here is another list, perhaps a grocery list:\n\n\\begin{enumerate}\n  \\setcounter{enumi}{3}\n  \\item apples\n  \\item bananas\n  \\item milk\n\\end{enumerate}\n\nLalalala\n\n'}]


In [7838]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_2'
file = dir / 'main.tex'
sample_latex_text = text_from_file(file)
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(document, dir)
print(parts)

[{'note_title': '1', 'text': '\\maketitle'}, {'note_title': 'abstract', 'text': '\\begin{abstract}\nThis is an abstract\n\\end{abstract}'}]


The `divide_latex_text` function by default divides along a LaTeX environment (something which is invoked by `\begin{...} \end{...}`). One can use the optional `environments_to_not_divide_along` parameter in the function to specify which environments to not divide along. By default, this list is set as follows:

In [7839]:
DEFAULT_ENVIRONMENTS_TO_NOT_DIVIDE_ALONG

['align',
 'align*',
 'center',
 'diagram',
 'displaymath',
 'displaymath*',
 'enumerate',
 'eqnarray',
 'eqnarray*',
 'equation',
 'equation*',
 'gather',
 'gather*',
 'itemize',
 'label',
 'multiline',
 'multiline*',
 'multline',
 'multline*',
 'proof',
 'quote',
 'tabular',
 'table']

In the following example, the `theorem`, `corollary`, and `definition` environments share a counter, which is not reset even when a new section begins. 

In [7840]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_1_consecutive_numbering_scheme'
file = dir / 'main.tex'
text = text_from_file(file)
print(text)

\documentclass{article}
\usepackage{amsthm}
%\usepackage{amsmath}

\newtheorem{theorem}{Theorem}
\newtheorem{corollary}[theorem]{Corollary}
\newtheorem{definition}[theorem]{Definition}
\newtheorem*{remark*}{Remark}

%\numberwithin{theorem}{part}

\begin{document}
For this document, the `theorem` counter is not reset whenever a new section begins.

A similar numbering scheme can be accomplished by importing \verb|amsmath| and invoking the code \verb|\numberwithin{theorem}{part}| in the preamble.

\section{Introduction}

\begin{theorem}
This is Theorem 1.
\end{theorem}

\begin{corollary}
This is Corollary 2.
\end{corollary}

\begin{remark*}
This is a remark. It is unnumbered and it does not affect the numberings of other environments.
\end{remark*}

\begin{definition}
This is Definition 3.
\end{definition}



\section{Another Section}

\begin{theorem}
This is Theorem 4.
\end{theorem}

And we might get a corollary!

\begin{corollary}
This is Corollary 5.
\end{corollary}

\begin{definition

In [7841]:
sample_output = divide_latex_text(text, dir)
sample_output

[{'note_title': '1',
  'text': 'For this document, the `theorem` counter is not reset whenever a new section begins.\n\nA similar numbering scheme can be accomplished by importing \\verb|amsmath| and invoking the code \\verb|\\numberwithin{theorem}{part}| in the preamble.'},
 {'note_title': '1. Introduction', 'text': '\\section{Introduction}\n\n'},
 {'note_title': 'Theorem 1.',
  'text': '\\begin{theorem}\nThis is Theorem 1.\n\\end{theorem}'},
 {'note_title': 'Corollary 2.',
  'text': '\\begin{corollary}\nThis is Corollary 2.\n\\end{corollary}'},
 {'note_title': 'Remark',
  'text': '\\begin{remark*}\nThis is a remark. It is unnumbered and it does not affect the numberings of other environments.\n\\end{remark*}'},
 {'note_title': 'Definition 3.',
  'text': '\\begin{definition}\nThis is Definition 3.\n\\end{definition}'},
 {'note_title': '2. Another Section',
  'text': '\\section{Another Section}\n\n'},
 {'note_title': 'Theorem 4.',
  'text': '\\begin{theorem}\nThis is Theorem 4.\n\\end{

In [7842]:
assert sample_output[0]['note_title'] == '1'
assert sample_output[1]['note_title'] == '1. Introduction'
assert sample_output[2]['note_title'] == 'Theorem 1.'
assert sample_output[3]['note_title'] == 'Corollary 2.'
assert sample_output[4]['note_title'] == 'Remark'

In the following example, the `\numerwithin` command is used to make the theorem-like environments numbered within sections. These environments are first numbered `1.1`, `1.2`, `1.3`, etc., and then numbered `2.1`, `2.2`, `2.3`, etc. once a new section starts.

In [7843]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_2_numbering_scheme_reset_at_each_section'
file = dir / 'main.tex'
text = text_from_file(file)
print(text)

% This is an example of a LaTeX document whose theorem-like environments are numbered with sections.

\documentclass{article}
\usepackage{amsthm}
\usepackage{amsmath}

\newtheorem{theorem}{Theorem}
\newtheorem{corollary}[theorem]{Corollary}
\newtheorem{definition}[theorem]{Definition}
\newtheorem*{remark*}{Remark}

\numberwithin{theorem}{section}

\begin{document}

This document resets its `theorem` counter whenever a new section begins.

\section{Introduction}

\begin{theorem}
This is Theorem 1.1.
\end{theorem}

\begin{corollary}
This is Corollary 1.2.
\end{corollary}

\begin{remark*}
This is a remark. It is unnumbered and it does not affect the numberings of other environments.
\end{remark*}


\begin{definition}
This is Definition 1.3.
\end{definition}



\section{Another Section}

\begin{theorem}
This is Theorem 2.1.
\end{theorem}

\begin{corollary}
This is Corollary 2.2.
\end{corollary}

\begin{definition}
This is Definition 2.3.
\end{definition}

\end{document}



In [7844]:
divide_latex_text(text, dir)

[{'note_title': '1',
  'text': 'This document resets its `theorem` counter whenever a new section begins.'},
 {'note_title': '1. Introduction', 'text': '\\section{Introduction}\n\n'},
 {'note_title': 'Theorem 1.1.',
  'text': '\\begin{theorem}\nThis is Theorem 1.1.\n\\end{theorem}'},
 {'note_title': 'Corollary 1.2.',
  'text': '\\begin{corollary}\nThis is Corollary 1.2.\n\\end{corollary}'},
 {'note_title': 'Remark',
  'text': '\\begin{remark*}\nThis is a remark. It is unnumbered and it does not affect the numberings of other environments.\n\\end{remark*}'},
 {'note_title': 'Definition 1.3.',
  'text': '\\begin{definition}\nThis is Definition 1.3.\n\\end{definition}'},
 {'note_title': '2. Another Section',
  'text': '\\section{Another Section}\n\n'},
 {'note_title': 'Theorem 2.1.',
  'text': '\\begin{theorem}\nThis is Theorem 2.1.\n\\end{theorem}'},
 {'note_title': 'Corollary 2.2.',
  'text': '\\begin{corollary}\nThis is Corollary 2.2.\n\\end{corollary}'},
 {'note_title': 'Definition 2.

In this example, the various theorem-like environments share a counter with `equation` environments and this counter is reset at the start of each new section.

In [7845]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_3_theorem_like_environments_share_counter_with_equation_and_reset_at_each_section' 
file = dir / 'main.tex'
text = text_from_file(file)
print(text)

\documentclass{amsart}
\usepackage[utf8]{inputenc}
\usepackage{amsmath, amsfonts, amssymb, amsthm, amsopn}

\numberwithin{equation}{section}

\theoremstyle{plain}
\newtheorem*{theorem*}{Theorem}
\newtheorem*{theoremA}{Theorem A}
\newtheorem*{theoremB}{Theorem B}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem{proposition}[equation]{Proposition}
\newtheorem{lemma}[equation]{Lemma}
\newtheorem{corollary}[equation]{Corollary}

\theoremstyle{definition}
\newtheorem{definition}[equation]{Definition}
\newtheorem{example}[equation]{Example}
\newtheorem*{acknowledgements}{Acknowledgements}
\newtheorem*{conventions}{Conventions}

\theoremstyle{remark}
\newtheorem{remark}[equation]{Remark}

\begin{document}

\section{Introduction}

\begin{theorem}
This is Theorem 1.1. This is because the \verb|\numberwithin{equation}{section}| makes the section number included in the equation counter and because the \\
\verb|\newtheorem{theorem}[equation]{Theorem}| command makes the environment \verb|theorem

In [7846]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_3_theorem_like_environments_share_counter_with_equation_and_reset_at_each_section'
file = dir / 'main.tex'
text = text_from_file(file)
divide_latex_text(text, dir)

[{'note_title': '1. Introduction', 'text': '\\section{Introduction}\n\n'},
 {'note_title': 'Theorem 1.1.',
  'text': '\\begin{theorem}\nThis is Theorem 1.1. This is because the \\verb|\\numberwithin{equation}{section}| makes the section number included in the equation counter and because the \\\\\n\\verb|\\newtheorem{theorem}[equation]{Theorem}| command makes the environment \\verb|theorem| be counted by the equation counter.\n\\end{theorem}'},
 {'note_title': '1',
  'text': 'The following makes an equation labeled 1.2; \n\\begin{equation}\n5 + 7 = 12\n\\end{equation}'},
 {'note_title': 'Theorem',
  'text': '\\begin{theorem*}\nThis Theorem is unnumbered\n\\end{theorem*}'},
 {'note_title': 'Corollary 1.3.',
  'text': '\\begin{corollary}\nThis is Corollary 1.3.\n\\end{corollary}'},
 {'note_title': '2. Another section', 'text': '\\section{Another section}\n'},
 {'note_title': 'Theorem 2.1.',
  'text': '\\begin{theorem}\nThis is theorem 2.1\n\\end{theorem}'},
 {'note_title': '2',
  'text':

In [7847]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_4_unnumbered_section'
file = dir / 'main.tex'
text = text_from_file(file)
print(divide_latex_text(text, dir))

[{'note_title': '1. This is section 1', 'text': '\\section{This is section 1}\n\n'}, {'note_title': 'Theorem 1.1.', 'text': '\\begin{theorem}\nThis is Theorem 1.1.\n\\end{theorem}'}, {'note_title': '1.1. This is a subsection 1.1', 'text': '\\subsection{This is a subsection 1.1}\n\nThe following makes an equation labeled 1; \n\\begin{equation}\n5 + 7 = 12\n\\end{equation}\n\n'}, {'note_title': 'Theorem', 'text': '\\begin{theorem*}\nThis Theorem is unnumbered\n\\end{theorem*}'}, {'note_title': '1.2. This is subsection 1.2', 'text': '\\subsection{This is subsection 1.2}\n\n'}, {'note_title': 'Corollary 1.2.', 'text': '\\begin{corollary}\nThis is Corollary 1.2.\n\\end{corollary}'}, {'note_title': 'Unnumbered section', 'text': '\\section*{Unnumbered section}\n\n'}, {'note_title': '1.3. This is subsection 1.3', 'text': '\\subsection{This is subsection 1.3}\n'}, {'note_title': '1.3.1. This is subsubsection 1.3.1', 'text': '\\subsubsection{This is subsubsection 1.3.1}\n\n'}, {'note_title': 'Th

In [7848]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_5_subsections_and_theorem_like_environments_share_counter'
file = dir / 'main.tex'
text = text_from_file(file)
sample_output = divide_latex_text(text, dir)
print(divide_latex_text(text, dir))
test_eq(sample_output[4]['note_title'], '1. Remark.')
test_eq(sample_output[5]['note_title'], 'Remark')

[{'note_title': '1. This is section 1', 'text': '\\section{This is section 1}\n\n'}, {'note_title': '1.1. Theorem.', 'text': '\\begin{thm}\nThis is 1.1. Theorem. Note that the \\verb|\\swapnumbers| command is invoked in the preamble.\n\\end{thm}'}, {'note_title': '1.2. This is 1.2. subsection.', 'text': '\\subsection{This is 1.2. subsection.}\n\nNote that the equation counter is numbered within the subsection counter and that the theorem-like environments are numbered with the equation counter.\n\n'}, {'note_title': '1.2.1. This is 1.2.1. Subsubsection', 'text': '\\subsubsection{This is 1.2.1. Subsubsection}\n\n'}, {'note_title': '1. Remark.', 'text': '\\begin{remark}\nThis is an 1. Remark. Note that \\verb|\\remark| has a counter separate from those of many of the other theorem-like environments.\n\\end{remark}'}, {'note_title': 'Remark', 'text': '\\begin{rem*}\nThis is an unnumbered Remark.\n\\end{rem*}'}, {'note_title': '1.3. Proposition.', 'text': '\\begin{prop}\nThis is 1.3. Propo

In the below example, the `theorem` count is specified to reset at every new section and the `corollary` environment is specified to reset at every new theorem.

In particular, note that there is a Theorem 1.2 and a subsequent Corollary 1.2.1 in the example:

In [7849]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_7_newtheorem_command_restarts_counter_by_section'
file = dir / 'main.tex'
text = text_from_file(file) 
print(text)
sample_output = divide_latex_text(text, dir )
print(divide_latex_text(text, dir))
test_eq(sample_output[4]['note_title'], 'Corollary 1.2.1.')


% Based on an example from https://www.overleaf.com/learn/latex/Theorems_and_proofs#Numbered_theorems.2C_definitions.2C_corollaries_and_lemmas

\documentclass[12 pt]{amsart}

\newtheorem{theorem}{Theorem}[section]
\newtheorem{corollary}{Corollary}[theorem]
\newtheorem{lemma}[theorem]{Lemma}
% Note that the below invocation of \newtheorem is invalid:
% \newtheorem{proposition}[theorem]{Proposition}[section]
\newtheorem{proposition}{Proposition}[section]

\begin{document}
\section{Introduction}
Theorems can easily be defined:

\begin{theorem}
Let \(f\) be a function whose derivative exists in every point, then \(f\) is 
a continuous function.
\end{theorem}

\begin{theorem}[Pythagorean theorem]
\label{pythagorean}
This is a theorem about right triangles and can be summarised in the next 
equation 
\[ x^2 + y^2 = z^2 \]
\end{theorem}

And a consequence of theorem \ref{pythagorean} is the statement in the next 
corollary.

\begin{corollary}
There's no right rectangle whose sides measure 3c

Note that part titles are stripped and are single-lined:

In [7850]:
# TODO: fill in the following example
# part = parts[...]
# assert part[0].strip() == part[0]

In the following example, the subsections and the theorem-like environments share a counter:


In [7851]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_8_subsubsections_and_theorems_share_counter'
file = dir / 'main.tex'
text = text_from_file(file) 
print(text)
sample_output = divide_latex_text(text, dir)
print(sample_output)
test_eq(sample_output[-1]['note_title'], 'Theorem 1.1.2.')
test_eq(sample_output[-2]['note_title'], '1.1.1. section 1.1.1')


% Based on an example from https://www.overleaf.com/learn/latex/Theorems_and_proofs#Numbered_theorems.2C_definitions.2C_corollaries_and_lemmas

\documentclass[12 pt]{amsart}

\newtheorem{cor}[subsubsection]{Corollary}
\newtheorem{lem}[subsubsection]{Lemma}
\newtheorem{prop}[subsubsection]{Proposition}
\newtheorem{propconstr}[subsubsection]{Proposition-Construction}
\newtheorem{lemconstr}[subsubsection]{Lemma-Construction}
\newtheorem{ax}[subsubsection]{Axiom}
\newtheorem{conj}[subsubsection]{Conjecture}
\newtheorem{thm}[subsubsection]{Theorem}
\newtheorem{qthm}[subsubsection]{Quasi-Theorem}
\newtheorem{qlem}[subsubsection]{Quasi-Lemma}
\newtheorem{defn}[subsubsection]{Definition}
\newtheorem{quest}[subsubsection]{Question}
\newtheorem{claim}[subsubsection]{Claim}

\begin{document}
\section{Introduction}

\subsection{section 1.1}

\subsubsection{section 1.1.1}

\begin{thm}
This is theorem 1.1.2
\end{thm}

\end{document}


[{'note_title': '1. Introduction', 'text': '\\section{Introducti

In the below example, note that there is some text immediately following the subsubsection; the "part" for the start of the subsubsection is joined by this following text: 

In [7852]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_text_after_subsubsection'
file = dir / 'main.tex'
sample_latex_text = text_from_file(file)
print(sample_latex_text)
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(sample_latex_text, dir)
print(parts)
test_eq(parts[2], DividedLatexPart(note_title='1.1.1. section 1.1.1', text='\\subsubsection{section 1.1.1}\nSome text beneath subsubsection\n'))


% Based on an example from https://www.overleaf.com/learn/latex/Theorems_and_proofs#Numbered_theorems.2C_definitions.2C_corollaries_and_lemmas

\documentclass[12 pt]{amsart}

\newtheorem{cor}[subsubsection]{Corollary}
\newtheorem{lem}[subsubsection]{Lemma}
\newtheorem{prop}[subsubsection]{Proposition}
\newtheorem{propconstr}[subsubsection]{Proposition-Construction}
\newtheorem{lemconstr}[subsubsection]{Lemma-Construction}
\newtheorem{ax}[subsubsection]{Axiom}
\newtheorem{conj}[subsubsection]{Conjecture}
\newtheorem{thm}[subsubsection]{Theorem}
\newtheorem{qthm}[subsubsection]{Quasi-Theorem}
\newtheorem{qlem}[subsubsection]{Quasi-Lemma}
\newtheorem{defn}[subsubsection]{Definition}
\newtheorem{quest}[subsubsection]{Question}
\newtheorem{claim}[subsubsection]{Claim}

\begin{document}
\section{Introduction}

\subsection{section 1.1}

\subsubsection{section 1.1.1}
Some text beneath subsubsection
\begin{thm}
This is theorem 1.1.2
\end{thm}

\end{document}


[{'note_title': '1. Introduction'

In the below example, theorem-like environments and equation environments share a counter and there is an equation within a theorem:

In [7853]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_theorems_and_equations_share_counter_and_equation_in_theorem'
file = dir / 'main.tex'
sample_latex_text = text_from_file(file)
print(sample_latex_text)
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(sample_latex_text, dir)
print(parts)
test_eq(parts[2], DividedLatexPart(note_title='Corollary 1.3.', text='\\begin{cor}\nThis is Corollary 1.3\n\\end{cor}'))


\documentclass[12pt]{amsart}
\usepackage{amsmath}
\usepackage{amsfonts}


\numberwithin{equation}{section}
\numberwithin{figure}{section}

\newtheorem{lemma}[equation]{Lemma}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem{conjecture}[equation]{Conjecture}
\newtheorem{cor}[equation]{Corollary}
\newtheorem{prop}[equation]{Proposition}

\begin{document}

\section{Introduction}

\begin{theorem}
\begin{equation}
asdf
\end{equation}
\end{theorem}

\begin{cor}
This is Corollary 1.3
\end{cor}

\end{document}
[{'note_title': '1. Introduction', 'text': '\\section{Introduction}\n\n'}, {'note_title': 'Theorem 1.1.', 'text': '\\begin{theorem}\n\\begin{equation}\nasdf\n\\end{equation}\n\\end{theorem}'}, {'note_title': 'Corollary 1.3.', 'text': '\\begin{cor}\nThis is Corollary 1.3\n\\end{cor}'}]


In the below example, theorem-like environments and equation environments share a counter and there is an equation within the proof of a proposition:

In [7854]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_equation_in_proof'
file = dir / 'main.tex'
sample_latex_text = text_from_file(file)
print(sample_latex_text)
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(sample_latex_text, dir)
print(parts)
test_eq(parts[2], DividedLatexPart(note_title='Corollary 1.3.', text='\\begin{cor}\nThis is Corollary 1.3\n\\end{cor}'))


\documentclass[12pt]{amsart}
\usepackage{amsmath}
\usepackage{amsfonts}


\numberwithin{equation}{section}
\numberwithin{figure}{section}

\newtheorem{lemma}[equation]{Lemma}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem{conjecture}[equation]{Conjecture}
\newtheorem{cor}[equation]{Corollary}
\newtheorem{prop}[equation]{Proposition}

\begin{document}

\section{Introduction}

\begin{prop}
This is Proposition 1.1
\end{prop}
\begin{proof}
\begin{equation}
\end{equation}
\end{proof}

\begin{cor}
This is Corollary 1.3
\end{cor}

\end{document}
[{'note_title': '1. Introduction', 'text': '\\section{Introduction}\n\n'}, {'note_title': 'Proposition 1.1.', 'text': '\\begin{prop}\nThis is Proposition 1.1\n\\end{prop}\\begin{proof}\n\\begin{equation}\n\\end{equation}\n\\end{proof}'}, {'note_title': 'Corollary 1.3.', 'text': '\\begin{cor}\nThis is Corollary 1.3\n\\end{cor}'}]


In the below example, theorem-like environments and equation environments share a counter and there is an `eqnarray` after the start of a section:

In [7855]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_eqnarray_after_start_of_section'
file = dir / 'main.tex'
sample_latex_text = text_from_file(file)
print(sample_latex_text)
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(sample_latex_text, dir)
print(parts)
test_eq(parts[1], DividedLatexPart(note_title='Proposition 1.2.', text='\\begin{prop}\nThis is Proposition 1.2\n\\end{prop}'))


\documentclass[12pt]{amsart}
\usepackage{amsmath}
\usepackage{amsfonts}


\numberwithin{equation}{section}
\numberwithin{figure}{section}

\newtheorem{lemma}[equation]{Lemma}
\newtheorem{theorem}[equation]{Theorem}
\newtheorem{conjecture}[equation]{Conjecture}
\newtheorem{cor}[equation]{Corollary}
\newtheorem{prop}[equation]{Proposition}

\begin{document}

\section{Introduction}
lalalala some stuff $5$

Hello I am saying stuff
\begin{eqnarray}
\end{eqnarray}
\begin{prop}
This is Proposition 1.2
\end{prop}

\end{document}
[{'note_title': '1. Introduction', 'text': '\\section{Introduction}\nlalalala some stuff $5$\n\nHello I am saying stuff\n\\begin{eqnarray}\n\\end{eqnarray}\n'}, {'note_title': 'Proposition 1.2.', 'text': '\\begin{prop}\nThis is Proposition 1.2\n\\end{prop}'}]


In the below example, there are many custom commands deifned using the `\def` command. Note that use of `\u` in the LaTeX file, which causes problems for `pylatexenc`. 

The `divide_latex_text` function had difficulties parsing through the latex document for the below example, specifically because this `\u` command made `pylatexenc` unable to find the `\section` following the `\u`. Now that `divide_latex_text` provides the option to replace custom commands from the LaTeX document (with their underlying "meaning/definitions" via the `replace_commands_in_latex_document` function) before parsing through the LaTeX document, this is no longer a problem.

In [7856]:
dir = _test_directory() / 'latex_examples' / 'divide_latex_example_unknown_section_division_problem'
file = dir / 'main.tex'
sample_latex_text = text_from_file(file)
# print(sample_latex_text)
preamble, document = divide_preamble(sample_latex_text)
parts = divide_latex_text(sample_latex_text, dir)
# test_eq(parts[1], ['Proposition 1.2.', '\\begin{prop}\nThis is Proposition 1.2\n\\end{prop}'])
test_eq(len(parts), 3)
print(parts)

[{'note_title': '1. Background and Notation', 'text': '\\section{Background and Notation}\n\n'}, {'note_title': '1.1. Unitary groups', 'text': '\\subsection{Unitary groups}\n\\label{subsecunitary}\n\nwhere $\\bm{{\\rm R}}_{{\\mathcal O}_E/{\\mathbb Z}}$ is the restriction of scalars functor.\nThen $SU$ is the derived group of $GU$ and of ${\\rm U}$,\n\n'}, {'note_title': '2. Mumford-Tate groups and endomorphism rings', 'text': '\\section{Mumford-Tate groups and endomorphism rings}\n\n\\label{secmt}\n\nCarlson and Toledo have \n\n\\bibliographystyle{hamsplain}\n\\bibliography{jda}\n\n'}]


In [7857]:
# TODO: example with a multilined section title forced to single-lined:
# e.g. `\section{Exceptional maximal subgroups of 
# \texorpdfstring{\(\GSp_4(\ff_\ell)\)}{GSp4Fell}}`

In [7858]:
# TODO: Find a list of environment names commonly used.

In [7859]:
# TODO: examples with different numbering convention and different numbered environments

In [7860]:
# Here are some latex files with different conventions:
# - Different environment types have different counts and the counts do not show the section number.
#   - vankataramana_imbrd https://arxiv.org/abs/1205.6543: 
#       - e.g. section 1 has Theorem 1, Remark 1, Remark 2, Remark 3, subsection 1.1.3 has Remark 4, Subsection 2.2 has Definition 1

In the following example, we demonstrate the hierarchical numbering fix. Even when a theorem is defined with [section], the \numberwithin{theorem}{subsection} command in the preamble overrides it, creating a deep 1.1.1 chain.

In [7861]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_9_hierarchical_override'
file = dir / 'main.tex'
text = text_from_file(file)

sample_output = divide_latex_text(text, dir)

# Verification based on your print output:
test_eq(len(sample_output), 4)
test_eq(sample_output[0]['note_title'], '1. First Section')
test_eq(sample_output[1]['note_title'], '1.1. First Subsection')
test_eq(sample_output[2]['note_title'], 'Theorem 1.1.1.')
test_eq(sample_output[3]['note_title'], 'Theorem 1.1.2.') # Changed from index 4 to 3

In the below example, we verify the recursive reset logic. When a new \section starts, it must reset the subsection counter, which in turn must reset any environments numbered within the subsection (like theorem or equation).

In [7862]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_10_recursive_cascading_reset'
file = dir / 'main.tex'
text = text_from_file(file)
print(text)
sample_output = divide_latex_text(text, dir)

# Check Section 1
test_eq(sample_output[2]['note_title'], 'Theorem 1.1.1.')
# After a new \section{Conclusion}, the subsection and theorem should both reset to 1
# ensuring we don't get "Theorem 2.1.2" or "Theorem 2.0.1"
test_eq(sample_output[5]['note_title'], 'Theorem 2.1.1.')

\documentclass{article}
\usepackage{amsmath, amsthm}

\newtheorem{theorem}{Theorem}[section]
\numberwithin{theorem}{subsection}

\begin{document}

\section{Introduction}
\subsection{Basics}

\begin{theorem}
First theorem of the first subsection. (1.1.1)
\end{theorem}

\section{Conclusion}
\subsection{Summary}

\begin{theorem}
This must reset to 2.1.1, not 2.1.2 or 2.0.1.
\end{theorem}

\end{document}


In this example, we see how the divide_latex_text function handles a mixed numbering scheme where some environments are numbered globally, while others are nested. The new implementation correctly climbs the hierarchy only for the environments that require it.

In [7863]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_11_mixed_hierarchy'
file = dir / 'main.tex'
text = text_from_file(file)
print(text)
sample_output = divide_latex_text(text, dir)

# Part 0: \section{Analysis} -> "1. Analysis"
# Part 1: \subsection{Methodology} -> "1.1. Methodology"
# Part 2: \begin{thm} -> "Theorem 1.1.1."
# Part 3: \begin{remark} -> "Remark 1."

test_eq(sample_output[2]['note_title'], 'Theorem 1.1.1.') # Changed index and number
test_eq(sample_output[3]['note_title'], 'Remark 1.')      # Changed index

\documentclass{article}
\usepackage{amsmath, amsthm}

\newtheorem{thm}{Theorem}[section]
\numberwithin{thm}{subsection}
\newtheorem{remark}{Remark} % Global counter

\begin{document}

\section{Analysis}
\subsection{Methodology}

\begin{thm}
A nested theorem (1.1.1).
\end{thm}

\begin{remark}
A global remark (1).
\end{remark}

\end{document}


The following example demonstrates deep subsubsection nesting. The _title_for_section_subsection_subsubsection_node helper now dynamically builds the title string, supporting depths that were previously hardcoded.

In [7864]:
dir = _test_directory() / 'latex_examples' / 'numbering_example_12_deep_subsubsection_nesting'
file = dir / 'main.tex'
text = text_from_file(file)
print(text)
sample_output = divide_latex_text(text, dir)

# Part 0: \section{Homological Algebra} -> '1. Homological Algebra'
# Part 1: \subsection{Chain Complexes} -> '1.1. Chain Complexes'
# Part 2: \subsubsection{Deep Dive...}  -> '1.1.1. Deep Dive into Homological Algebra'
# Part 3: \subsubsection{Spectral Sequences} -> '1.1.2. Spectral Sequences'

test_eq(sample_output[2]['note_title'], '1.1.1. Deep Dive into Homological Algebra')
test_eq(sample_output[3]['note_title'], '1.1.2. Spectral Sequences')

\documentclass{article}

\begin{document}

\section{Homological Algebra}
\subsection{Chain Complexes}

\subsubsection{Deep Dive into Homological Algebra}
This is section 1.1.1.

\subsubsection{Spectral Sequences}
This is section 1.1.2.

\end{document}


In this example, we verify that the \@addtoreset{equation}{section} command is correctly parsed and that manual environments like 'remark' and 'example' correctly follow the Section.Equation numbering format (e.g., Remark 2.1).

In [7865]:

dir = _test_directory() / 'latex_examples' / 'numbering_example_13_manual_overrides'
file = dir / 'main.tex'
# The main.tex contains the preamble you provided and a body like:
# \section{Introduction}
# \begin{remark} First remark \end{remark}  --> Should be Remark 1.1
# \section{Main Results}
# \begin{example} First example \end{example} --> Should be Example 2.1

text = text_from_file(file)
sample_output = divide_latex_text(text, dir)

# Verification:
# Note: Part 0 is usually the section, Part 1 the environment
test_eq(sample_output[0]['note_title'], '1. Introduction')
test_eq(sample_output[1]['note_title'], 'Remark 1.1.')
test_eq(sample_output[2]['note_title'], '2. Main Results')
test_eq(sample_output[3]['note_title'], 'Example 2.1.')

Here we test the deep hierarchy: \@addtoreset{section}{part} and \@addtoreset{equation}{section}. The title should dynamically climb to the Part level if that counter is active.

In [7866]:

dir = _test_directory() / 'latex_examples' / 'numbering_example_14_part_section_cascade'
file = dir / 'main.tex'
# Content:
# \part{Algebra}
# \section{Modules}
# \begin{remark} ... \end{remark} --> Should be Remark 1.1.1

text = text_from_file(file)
sample_output = divide_latex_text(text, dir)

# Verification:
# We expect the numbering to reflect the Part.Section.Equation structure
test_eq(sample_output[2]['note_title'], 'Remark 1.1.1.')

This example tests the 'warning' environment defined in the preamble.  Even though it's a complex \newenvironment with \manfntsymbol, our logic should identify the \refstepcounter{equation} and provide the correct hierarchical title.

In [7867]:

dir = _test_directory() / 'latex_examples' / 'numbering_example_15_custom_warning_env'
file = dir / 'main.tex'
# Content:
# \section{Safety}
# \begin{warning} Dangerous math ahead. \end{warning}

text = text_from_file(file)
sample_output = divide_latex_text(text, dir)

# Verification:
test_eq(sample_output[1]['note_title'], 'Warning 1.1.')

This example verifies that commented-out \newtheorem commands are ignored while the subsequent \newenvironment definitions of the same name are correctly mapped to the equation counter.

In [7868]:

dir = _test_directory() / 'latex_examples' / 'numbering_example_16_commented_definitions'
file = dir / 'main.tex'
text = text_from_file(file)
sample_output = divide_latex_text(text, dir)

test_eq(sample_output[2]['note_title'], 'Remark 1.2.')
sample_output
# Ensure 'remark' uses the equation counter (linked to section) 
# rather than being treated as a global counter.

[{'note_title': '1. Logic Check', 'text': '\\section{Logic Check}\n'},
 {'note_title': 'Theorem 1.1.',
  'text': '\\begin{theorem}\nTheorem 1.1.\n\\end{theorem}'},
 {'note_title': 'Remark 1.2.',
  'text': '\\begin{remark}\nRemark 1.2.\n\\end{remark}'}]

This example confirms that when environments are manually defined using \newenvironment with a \refstepcounter{equation} command, the system correctly identifies equation as the underlying counter. It specifically ensures that these items increment consecutively with other environments sharing that counter (like theorems) and reset correctly when a new section begins.

In [7869]:
# Example 17: Custom environments manually linked to the equation counter
# This verifies that the logic detects \refstepcounter{equation} inside \newenvironment
# and respects the global equation numbering sequence.

dir = _test_directory() / 'latex_examples' / 'numbering_example_17_custom_equation_environments'
file = dir / 'main.tex'
text = text_from_file(file)

# The LaTeX contains:
# \newtheorem{theorem}[equation]{Theorem}
# \newenvironment{remark}{\refstepcounter{equation}...}{}
# \newenvironment{warning}{\refstepcounter{equation}...}{}

sample_output = divide_latex_text(text, dir)

# Test that the sequence is consecutive across different environment types
test_eq(sample_output[1]['note_title'], 'Theorem 1.1.')
test_eq(sample_output[2]['note_title'], 'Remark 1.2.')
test_eq(sample_output[3]['note_title'], 'Warning 1.3.')

sample_output
# Ensure the equation counter is correctly identified as the 'master' 
# for both standard theorems and custom-defined blocks.

[{'note_title': '1. Master Counter Test',
  'text': '\\section{Master Counter Test}\n\n'},
 {'note_title': 'Theorem 1.1.',
  'text': '\\begin{theorem}\nThis is the first item.\n\\end{theorem}'},
 {'note_title': 'Remark 1.2.',
  'text': '\\begin{remark}\nThis is the second item, using the same counter.\n\\end{remark}'},
 {'note_title': 'Warning 1.3.',
  'text': '\\begin{warning}\nThis is the third item.\n\\end{warning}'}]

In [7874]:
# Example 18: Equation environments as counter-incrementing nodes
# This verifies that actual math equation blocks correctly advance the 
# shared counter for subsequent theorem-like environments.

dir = _test_directory() / 'latex_examples' / 'numbering_example_18_interleaved_equations'
file = dir / 'main.tex'
text = text_from_file(file)

# The LaTeX defines \newtheorem{lemma}[equation]{Lemma}
# and \newenvironment{remark}{\refstepcounter{equation}...}{}

sample_output = divide_latex_text(text, dir)

# Test the interleaved sequence: 
# Section (resets to 0) -> Lemma (1.1) -> Equation (1.2) -> Remark (1.3)
test_eq(sample_output[1]['note_title'], 'Lemma 1.1.')
test_eq(sample_output[2]['note_title'], 'Remark 1.3.')
test_eq(sample_output[4]['note_title'], 'Lemma 1.5.')

sample_output
# Confirms that the equation counter is treated as a unified sequence
# regardless of whether it's an environment or a display math block.

[{'note_title': '1. Interleaved Numbering',
  'text': '\\section{Interleaved Numbering}\n\n'},
 {'note_title': 'Lemma 1.1.',
  'text': '\\begin{lemma}\nA Lemma that starts the count at 1.1.\n\\begin{equation}\ne^{i\\pi} + 1 = 0\n\\end{equation}\n\\end{lemma}'},
 {'note_title': 'Remark 1.3.',
  'text': '\\begin{remark}\nThis remark should be 1.3 because the equation above took the 1.2 slot.\n\\end{remark}'},
 {'note_title': '1', 'text': '\\begin{eqnarray}\n   asdf\n\\end{eqnarray}'},
 {'note_title': 'Lemma 1.5.',
  'text': '\\begin{lemma}\n   This lemma should be 1.5\n\\end{lemma}'}]